# Setup







## Imports

In [ ]:
import pandas as pd
import numpy as np

import warnings


from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, LassoCV, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from pathlib import Path

from sklearn.model_selection import train_test_split

import xgboost

from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

from scipy.stats import binomtest

## File paths

In [ ]:
output_path = Path("data/output/06_output")

prediction_output = Path("data/output/prediction_output")
prediction_output.mkdir(parents=True, exist_ok=True)

In [ ]:
summary_metrics_06 = pd.read_csv(output_path / "summary_metrics_06.csv", sep="|", na_values=[" ", ""],)

In [ ]:
output_path_08 = Path("data/output/08_output")
summary_metrics_08 = pd.read_csv(
    output_path_08 / "summary_metrics_08.csv", sep="|", na_values=[" ", ""],
)

## Shared setup (symptomatic-side derivation and bilateral pairs)

In [ ]:
def build_activity_scaling_preprocessor(
    *,
    demographic_covariate_columns: list[str],
    activity_predictor_columns: list[str],
) -> ColumnTransformer:
    """
    Build a column transformer that standardises the activity predictors
    while passing the demographic covariates through unscaled.

    Listing the covariate passthrough first and the scaled activity block
    second fixes the output column order to
    ``covariates + activity``, matching the ordering the coefficient
    extraction logic relies on.

    :param demographic_covariate_columns: Covariate column names passed
        through without scaling.
    :param activity_predictor_columns: Activity predictor column names
        standardised within each cross-validation fold.
    :returns: Configured, unfitted column transformer.
    """
    return ColumnTransformer(
        transformers=[
            ("covariates_unscaled", "passthrough", demographic_covariate_columns),
            ("activity_standardised", StandardScaler(), activity_predictor_columns),
        ],
        remainder="drop",
    )

# 1. Predictive models (Visit 06)

## 1.1 Prepare modelling dataframe
Derives the columns required by the modeling pipeline from the summary_metrics_06 dataframe.

Derivations:
- kl_grade_index_knee: worse-knee KL grade, used as train/test stratification key. Not used as a covariate.
- sex_female: binary sex covariate derived from P02SEX (1 = female, 0 = male).
- Symptomatic-side outcome columns: for bilateral knee variables, the more symptomatic knee is selected based on the outcome value.

In [ ]:
STRUCTURAL_COVARIATE_COLUMN = "kl_grade_index_knee"

DEMOGRAPHIC_COVARIATE_COLUMNS = [
    "V06AGE",
    "V06BMI",
    "V06COMORB",
    "sex_female",
]

FINAL_PREDICTOR_COLUMNS = [
    # Average daily activity counts (intensity levels)
    "V06AACNT",                                   # average daily counts
    "V06AALTMNT",                                 # average daily light activity counts Trioano
    "V06AAMDMNT",                                 # average daily moderate activity counts Trioano
    "V06AAMVMNT",                                 # average daily moderate/vigorous activity counts Trioano
    "V06AAVMNT",                                  # average daily vigorous activity counts Trioano

    # Onset / offset
    "activity_onset_minute_mean",                # Mean onset time across valid days
    "activity_onset_minute_sd",                  # Standard deviation of onset time
    "activity_offset_minute_mean",               # Mean offset time across valid days
    "activity_offset_minute_sd",                 # Standard deviation of offset time
    "wear_duration_mean",                        # Mean daily wear duration in minutes

    # Bout structure: sedentary
    "mean_sedentary_bout_count",
    "mean_sedentary_bout_mean_duration",
    "mean_sedentary_bout_max_duration",
    "mean_sedentary_bout_total_minutes",

    # Bout structure: light
    "mean_light_bout_mean_duration",
    "mean_light_bout_max_duration",
    "mean_light_bout_total_minutes",

    # Bout structure: MVPA
    "mean_mvpa_bout_count",
    "mean_mvpa_bout_mean_duration",

    # Harmonic features: overall
    "acrophase",                                 # Timing of the fitted rhythm's peak
    "interdaily_stability",                      # Day-to-day consistency of the rhythm relative to its 24h period

    # Harmonic features: day-type specific
    "mesor_mean_curve_weekday",                  # Mean activity level on weekdays
    "mesor_mean_curve_weekend",                  # Mean activity level on weekends
    "acrophase_mean_curve_weekday",              # Timing of activity peak on weekdays
    "acrophase_mean_curve_weekend",              # Timing of activity peak on weekends

    # IV: day-type specific
    "iv_weekday",                                # Within-day fragmentation on weekdays
    "iv_weekend",                                # Within-day fragmentation on weekends
]

OUTCOME_GROUPS: dict[str, list[str]] = {
    "pain": [
        "V06KOOSKPR",   # Right knee: KOOS Pain Score, 0-100 (higher = better)
        "V06ICPTSKR",   # ICOAP Right knee: pain total score (higher = worse)
        "V06KOOSKPL",   # Left knee: KOOS Pain Score, 0-100 (higher = better)
        "V06ICPTSKL",   # ICOAP Left knee: pain total score (higher = worse)
    ],
    "function": [
        "V0620MPACE",   # 20m walk pace (m/s)
        "V06CSTIME1",    # repeated chair stand time
        "V06400MTIM",   # 400m walk total time (baseline only; excluded from longitudinal)
        "V06400MTR",   # 400-metre walk total metres walked (exploratory)
    ],
    "depression": [
        "V06CESD",      # CES-D depression scale total score
    ],
    "self_reported_function_symptoms": [
        "V06WOMADLR",   # Right knee: WOMAC Disability Score
        "V06WOMADLL",   # Left knee: WOMAC Disability Score
        "V06KOOSYMR",  # Right knee: KOOS Symptoms Score (distinct from pain and function)
        "V06KOOSYML",  # Left knee: KOOS Symptoms Score (distinct from pain and function)
        ],

    "participation": [
        "V06LLDIFST",   # LLDI Frequency, Total Score
        "V06LLDILST",   # LLDI Limitation, Total Score
    ],

    "quality_of_life": [
        "V06KOOSQOL",    # KOOS Quality of Life Score
    ],
}

# Execution

# summary_metrics_06 is loaded fresh from CSV with a default RangeIndex
# and ID already a column, so no index reset is normally needed. Guard
# it so ID is only pulled off the index when an upstream step actually
# left it there -- an unconditional reset_index() would inject a stray
# "index" column that rides through the V06/V08 merge as index_v06/_v08.
summary_metrics_06 = summary_metrics_06.copy()
if "ID" not in summary_metrics_06.columns:
    summary_metrics_06 = summary_metrics_06.reset_index()

print(
    f"Loaded from memory:  {summary_metrics_06.shape[0]:,} participants, "
    f"{summary_metrics_06.shape[1]} columns"
)

kl_missing = summary_metrics_06[STRUCTURAL_COVARIATE_COLUMN].isna().sum()
print(f"{STRUCTURAL_COVARIATE_COLUMN} ready — missing in {kl_missing} participants")

# Derive a binary sex covariate from the OAI P02SEX item.
# P02SEX is coded 1 = Male, 2 = Female in OAI. Some data exports carry
# these as string labels rather than integers, so both encodings are
# mapped. The derived sex_female column uses 1 = female, 0 = male.
sex_mapping = {
    1.0: 0.0,
    2.0: 1.0,
    "1": 0.0,
    "2": 1.0,
    "1: Male": 0.0,
    "2: Female": 1.0,
    "M": 0.0,
    "F": 1.0,
    "Male": 0.0,
    "Female": 1.0,
}

print("\nRaw P02SEX values:")
print(summary_metrics_06["P02SEX"].value_counts(dropna=False).to_string())

summary_metrics_06["sex_female"] = summary_metrics_06["P02SEX"].map(sex_mapping)

unmapped_sex_count = (
    summary_metrics_06["sex_female"].isna().sum()
    - summary_metrics_06["P02SEX"].isna().sum()
)
if unmapped_sex_count > 0:
    print(
        f"WARNING — {unmapped_sex_count} P02SEX values did not match the "
        f"mapping and became NaN. Inspect the raw values printed above and "
        f"extend sex_mapping before continuing."
    )

sex_counts = summary_metrics_06["sex_female"].value_counts(dropna=False)
print(
    f"sex_female derived:\n"
    f"  Female (1)  : {sex_counts.get(1.0, 0)}\n"
    f"  Male   (0)  : {sex_counts.get(0.0, 0)}\n"
    f"  Missing     : {summary_metrics_06['sex_female'].isna().sum()}"
)

# Fail loudly if any modelled outcome is missing from the dataframe.
# Guards against a name in OUTCOME_GROUPS that is not present as a column
# in summary_metrics_06 (e.g. not loaded from the CSV, not merged in
# above, or a typo), which would otherwise surface only as a KeyError
# deep in a later stage.
grouped_outcomes = {
    outcome for outcomes in OUTCOME_GROUPS.values() for outcome in outcomes
}
underived_outcomes = grouped_outcomes - set(summary_metrics_06.columns)
assert not underived_outcomes, (
    f"OUTCOME_GROUPS references outcomes that are not present as derived "
    f"columns in summary_metrics_06: {sorted(underived_outcomes)}"
)

# Verify that all predictor columns are present. Missing columns here will cause silent NaN propagation or KeyErrors in later stages, so fail loudly now.
missing_predictors = [
    column for column in FINAL_PREDICTOR_COLUMNS
    if column not in summary_metrics_06.columns
]

if missing_predictors:
    print(
        f"WARNING — the following predictor columns are missing from the "
        f"dataframe and will cause errors in later stages:\n"
        f"  {missing_predictors}"
    )
else:
    print(f"All {len(FINAL_PREDICTOR_COLUMNS)} predictor columns present.")

# Final shape and missingness summary.
print(
    f"\nModelling dataframe ready:\n"
    f"  Shape         : {summary_metrics_06.shape}\n"
    f"  Participants  : {len(summary_metrics_06):,}"
)

predictor_missingness = (
    summary_metrics_06[FINAL_PREDICTOR_COLUMNS]
    .isna()
    .sum()
    .sort_values(ascending=False)
)
predictors_with_missing = predictor_missingness[predictor_missingness > 0]

if predictors_with_missing.empty:
    print("  Predictor missingness: none")
else:
    print("  Predictor missingness (columns with any NaN):")
    for column_name, missing_count in predictors_with_missing.items():
        print(
            f"    {column_name:<45} {missing_count:>4} "
            f"({100 * missing_count / len(summary_metrics_06):.1f} %)"
        )

## 1.2 Model families

### 1.2.1 LASSO

**Purpose**
LASSO (Least Absolute Shrinkage and Selection Operator) is used here in its explanatory role rather than as a pure predictive model. The key result is the incremental R², how much additional variance in each outcome is explained by the activity predictor block on top of the demographic and structural covariates alone.

**Design.**
The model is fitted in two blocks: Block 1 (baseline): demographic covariates only (age, BMI, comorbidity count, sex), fitted with plain OLS. Block 2 (full): covariates (unregularised) + activity predictors (standardised and L1-regularised via LassoCV).
The incremental R² (full minus baseline, both evaluated via 10-fold cross-validation on the training set) is the primary reported metric. It quantifies the unique contribution of the accelerometry-derived activity features independent of clinical background characteristics.

Held-out test set performance is not the focus of this stage. The predictive model families (Ridge, Elastic Net, Random Forest, XGBoost) in Stages 2–5 are evaluated on the held-out test set. LASSO is evaluated on the training set via cross-validation to preserve its explanatory interpretation.

In [ ]:
def prepare_modelling_dataset_for_stage(
    *,
    dataframe: pd.DataFrame,
    outcome_column: str,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    stratification_column: str,
    test_set_fraction: float = 0.20,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    Prepare a clean train/test split for a single outcome variable.

    Rows with any missing value in the covariate, predictor, or outcome
    columns are dropped before splitting. The split is stratified on KL
    grade (binned into three groups: 0-1, 2-3, 4) to preserve the
    severity distribution across both sets. KL grade is used only to
    filter complete cases and to stratify the split; it is not returned
    as a predictor.

    :param dataframe: Modelling dataframe with one row per participant.
    :param outcome_column: Name of the outcome variable to predict.
    :param activity_predictor_columns: Activity predictor column names.
    :param demographic_covariate_columns: Demographic covariate column
        names included without regularisation.
    :param stratification_column: KL grade column name used to stratify
        the train/test split. Included in the complete-case filter but
        never returned as a predictor.
    :param test_set_fraction: Proportion of data held out for testing.
    :param random_state: Random seed for reproducibility.
    :returns: Tuple of (training_predictors, training_outcome,
        test_predictors, test_outcome).
    :raises ValueError: If fewer than 100 complete cases remain after
        dropping missing values.
    """
    predictor_columns = list(demographic_covariate_columns) + activity_predictor_columns

    # KL grade is retained in the complete-case filter so it can drive
    # stratification, but it is not part of predictor_columns and is
    # therefore never passed to the model.
    columns_required_for_complete_cases = predictor_columns + [outcome_column]
    if stratification_column not in columns_required_for_complete_cases:
        columns_required_for_complete_cases = (
            columns_required_for_complete_cases + [stratification_column]
        )

    complete_cases = dataframe[columns_required_for_complete_cases].dropna()

    complete_case_count = len(complete_cases)
    total_count = len(dataframe)

    print(
        f"  Complete cases : {complete_case_count:,} of {total_count:,} "
        f"({100 * complete_case_count / total_count:.1f} %)"
    )

    if complete_case_count < 100:
        raise ValueError(
            f"Only {complete_case_count} complete cases remain for "
            f"'{outcome_column}'. Modelling would be unreliable."
        )

    outcome_series = complete_cases[outcome_column]
    predictor_dataframe = complete_cases[predictor_columns]

    kl_grade_bins = pd.cut(
        complete_cases[stratification_column],
        bins=[-0.5, 1.5, 3.5, 4.5],
        labels=["kl_0_1", "kl_2_3", "kl_4"],
    )

    (
        training_predictors,
        test_predictors,
        training_outcome,
        test_outcome,
    ) = train_test_split(
        predictor_dataframe,
        outcome_series,
        test_size=test_set_fraction,
        random_state=random_state,
        stratify=kl_grade_bins,
    )

    print(
        f"  Training set   : {len(training_predictors):,} participants\n"
        f"  Test set       : {len(test_predictors):,} participants"
    )

    return training_predictors, training_outcome, test_predictors, test_outcome

In [ ]:
def run_lasso_for_outcome(
    *,
    outcome_column: str,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    stratification_column: str,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit the two-block explanatory LASSO model for a single outcome and
    return all metrics in a result dictionary.

    Block 1 (baseline): demographic covariates only, plain OLS, evaluated
    via cross-validation to produce baseline_r2_cv.

    Block 2 (full): covariates (unregularised) concatenated with
    standardised activity predictors, regularised via LassoCV. Activity
    predictors are standardised within a pipeline so the scaler refits
    inside each cross-validation fold; validation rows never influence
    the scaling. Covariates pass through unscaled. Cross-validated R² on
    the training set produces full_r2_cv.

    The incremental R² (full_r2_cv minus baseline_r2_cv) is the primary
    reported metric: it quantifies the unique variance explained by the
    activity block after controlling for demographic covariates.

    :param outcome_column: Name of the outcome variable to model.
    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names
        (standardised within the pipeline and regularised).
    :param demographic_covariate_columns: Demographic covariate column
        names (included unregularised in both blocks).
    :param stratification_column: KL grade column used to stratify the
        train/test split; not returned as a predictor.
    :param cross_validation_folds: Number of CV folds (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys baseline_r2_cv, full_r2_cv,
        incremental_r2, selected_predictors, lasso_coefficients,
        optimal_alpha, outcome_column, n_training, n_test.
    """
    covariate_columns = list(demographic_covariate_columns)

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=outcome_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        stratification_column=stratification_column,
    )

    covariate_training = training_predictors[covariate_columns]

    # Block 1 — baseline: covariates only, plain OLS
    kfold = KFold(
        n_splits=cross_validation_folds,
        shuffle=True,
        random_state=random_state,
    )

    baseline_scores = cross_val_score(
        estimator=LinearRegression(),
        X=covariate_training,
        y=training_outcome,
        cv=kfold,
        scoring="r2",
    )
    baseline_r2_cv = float(baseline_scores.mean())

    # Block 2 — full: covariates (unscaled) + standardised activity
    # predictors, assembled in a pipeline so the scaler refits inside
    # each CV fold rather than being fitted once on the whole training set.
    full_pipeline = Pipeline(steps=[
        ("preprocess", build_activity_scaling_preprocessor(
            demographic_covariate_columns=covariate_columns,
            activity_predictor_columns=activity_predictor_columns,
        )),
        ("model", LassoCV(
            cv=cross_validation_folds,
            random_state=random_state,
            max_iter=50_000,
        )),
    ])

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        full_pipeline.fit(training_predictors, training_outcome)

        full_scores = cross_val_score(
            estimator=full_pipeline,
            X=training_predictors,
            y=training_outcome,
            cv=KFold(
                n_splits=cross_validation_folds,
                shuffle=True,
                random_state=random_state,
            ),
            scoring="r2",
        )
    full_r2_cv = float(full_scores.mean())
    incremental_r2 = full_r2_cv - baseline_r2_cv

    # Extract non-zero activity predictor coefficients.
    # build_activity_scaling_preprocessor lists the covariate passthrough
    # first and the activity block second, so the fitted coefficient vector
    # is ordered [covariates..., activity...]. The activity coefficients are
    # therefore those after the covariate block. This slice is only valid
    # while that column ordering holds — keep the two in sync.
    fitted_lasso = full_pipeline.named_steps["model"]
    optimal_alpha = float(fitted_lasso.alpha_)

    number_of_covariates = len(covariate_columns)
    activity_coefficients = fitted_lasso.coef_[number_of_covariates:]
    non_zero_mask = activity_coefficients != 0

    selected_predictors = [
        column
        for column, is_selected in zip(activity_predictor_columns, non_zero_mask)
        if is_selected
    ]

    lasso_coefficients = pd.Series(
        data=activity_coefficients[non_zero_mask],
        index=selected_predictors,
        name="lasso_coefficient",
    ).sort_values(key=abs, ascending=False)

    print(
        f"\n  Baseline R² (covariates only, {cross_validation_folds}-fold CV) : "
        f"{baseline_r2_cv:.4f}\n"
        f"  Full R²     (covariates + activity)            : "
        f"{full_r2_cv:.4f}\n"
        f"  Incremental R² (activity block)                : "
        f"{incremental_r2:.4f}\n"
        f"  Optimal alpha                                  : "
        f"{optimal_alpha:.6f}\n"
        f"  Predictors selected (non-zero)                 : "
        f"{len(selected_predictors)} / {len(activity_predictor_columns)}"
    )

    if selected_predictors:
        print("\n  Non-zero coefficients (ranked by absolute magnitude):")
        for predictor_name, coefficient_value in lasso_coefficients.items():
            print(f"    {predictor_name:<45} {coefficient_value:+.4f}")

    return {
        "outcome_column": outcome_column,
        "n_training": len(training_predictors),
        "n_test": len(test_predictors),
        "baseline_r2_cv": baseline_r2_cv,
        "full_r2_cv": full_r2_cv,
        "incremental_r2": incremental_r2,
        "selected_predictors": selected_predictors,
        "lasso_coefficients": lasso_coefficients,
        "optimal_alpha": optimal_alpha,
    }

#### Execute LASSO for all outcomes

In [ ]:
RANDOM_STATE = 42
CROSS_VALIDATION_FOLDS = 10

lasso_results: dict[str, list[dict]] = {}

for group_name, outcome_columns in OUTCOME_GROUPS.items():
    print(f"\n{'=' * 70}")
    print(f"Group: {group_name}  ({len(outcome_columns)} outcomes)")
    print(f"{'=' * 70}")

    group_results = []

    for outcome_column in outcome_columns:
        print(f"\n  Outcome: {outcome_column}")
        print(f"  {'-' * 50}")

        try:
            result = run_lasso_for_outcome(
                outcome_column=outcome_column,
                dataframe=summary_metrics_06,
                activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
                demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
                stratification_column=STRUCTURAL_COVARIATE_COLUMN,
                cross_validation_folds=CROSS_VALIDATION_FOLDS,
                random_state=RANDOM_STATE,
            )
            group_results.append(result)

        except ValueError as error:
            print(f"  Skipped: {error}")

    lasso_results[group_name] = group_results

#### Lasso summary table

In [ ]:
summary_rows = []

for group_name, group_results in lasso_results.items():
    for result in group_results:
        summary_rows.append({
            "group": group_name,
            "outcome": result["outcome_column"],
            "n_training": result["n_training"],
            "baseline_r2_cv": round(result["baseline_r2_cv"], 4),
            "full_r2_cv": round(result["full_r2_cv"], 4),
            "incremental_r2": round(result["incremental_r2"], 4),
            "n_selected_predictors": len(result["selected_predictors"]),
            "optimal_alpha": round(result["optimal_alpha"], 6),
        })

lasso_summary_table = pd.DataFrame(summary_rows)

print("\nStage 1 — LASSO summary (sorted by incremental R²):")
print(
    lasso_summary_table
    .sort_values("incremental_r2", ascending=False)
    .to_string(index=False)
)

lasso_summary_table.to_csv(
    prediction_output / "stage_1_lasso_summary.csv",
    index=False,
)

In [ ]:
def summarise_feature_selection_frequency(
    *,
    lasso_results_by_group: dict[str, list[dict]],
    all_activity_predictor_columns: list[str],
) -> pd.DataFrame:
    """
    Aggregate LASSO selection frequency across all modelled outcomes.

    For each activity predictor this reports how often it was retained
    (assigned a non-zero coefficient) across the successfully modelled
    outcomes, and the signed standardised coefficient it received for the
    flagship gait outcome. Aggregation is confined to the LASSO stage
    because LASSO was applied uniformly to every outcome with an identical
    predictor block; selection counts are therefore comparable across
    outcomes. Coefficients from different model families are deliberately
    not pooled, as signed standardised LASSO coefficients and tree gain
    importances are not on a common scale.

    Predictors that were never selected still appear, with a count of zero
    and a missing flagship coefficient, so the table covers the full
    predictor block rather than only the retained subset.

    :param lasso_results_by_group: Mapping from outcome-group name to the
        list of per-outcome result dictionaries produced by
        ``run_lasso_for_outcome``. Each result must contain the keys
        ``outcome_column``, ``selected_predictors`` and
        ``lasso_coefficients``.
    :param all_activity_predictor_columns: The complete activity predictor
        block offered to every outcome model. Defines the row set and
        guarantees never-selected predictors are still represented.
    :returns: One row per activity predictor, sorted by descending
        selection count and then by descending absolute flagship
        coefficient, with columns ``feature``, ``selection_count``,
        ``outcomes_modelled``, ``selection_proportion``,
        ``selected_for_flagship`` and ``flagship_signed_coefficient``.
    :raises ValueError: If no result matches ``flagship_outcome_column``.
    """
    all_results = [
        result
        for group_results in lasso_results_by_group.values()
        for result in group_results
    ]

    outcomes_modelled = len(all_results)

    selection_count_by_feature = {
        feature: 0 for feature in all_activity_predictor_columns
    }
    for result in all_results:
        for feature in result["selected_predictors"]:
            selection_count_by_feature[feature] += 1

    summary_rows = [
        {
            "feature": feature,
            "selection_count": selection_count_by_feature[feature],
            "outcomes_modelled": outcomes_modelled,
            "selection_proportion": round(
                selection_count_by_feature[feature] / outcomes_modelled, 4
            ),
        }
        for feature in all_activity_predictor_columns
    ]

    feature_selection_table = pd.DataFrame(summary_rows)

    return feature_selection_table.sort_values(
        by=["selection_count"],
        ascending=[False],
    ).reset_index(drop=True)

In [ ]:
feature_selection_table = summarise_feature_selection_frequency(
    lasso_results_by_group=lasso_results,
    all_activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
)

print("\nSection 2.4.5 — LASSO feature selection frequency:")
print(feature_selection_table.to_string(index=False))

feature_selection_table.to_csv(
    prediction_output / "stage_1_feature_selection_frequency.csv",
    index=False,
)

### 1.2.2 Ridge

**Purpose**

Ridge regression is the first purely predictive model in the pipeline.Unlike Stage 1 LASSO, the goal here is not feature selection or incremental explanation, it is to assess how well the full predictor set predicts each outcome on unseen data, evaluated on the held-out test set.

 **Regularisation**
 The regularisation strength alpha is selected via RidgeCV using 10-fold cross-validation on the training set across a log-spaced grid from 0.001 to 1000. Activity predictors are standardised on the training set; the same scaler is applied to the test set. Covariates are passed through unscaled, consistent with the convention used in Stage 1.


In [ ]:
def run_ridge_for_outcome(
    *,
    outcome_column: str,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    stratification_column: str,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit a Ridge regression model with cross-validated alpha selection
    for a single outcome and return held-out test set metrics.

    Activity predictors are standardised within a pipeline so the
    scaler refits inside each cross-validation fold; covariates pass
    through unscaled. Alpha is selected from a log-spaced grid of 25
    values between 0.001 and 1000 using RidgeCV with 10-fold CV on
    the training set.

    A second cross-validation pass using the selected alpha is run on
    the full training matrix to produce a cross_validated_r2 that is
    directly comparable to the baseline_r2_cv and full_r2_cv from
    Stage 1.

    :param outcome_column: Name of the outcome variable to model.
    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names
        (standardised before fitting).
    :param demographic_covariate_columns: Demographic covariate column
        names (passed through unscaled).
    :param stratification_column: KL grade column used to stratify the split; not returned as a predictor
    :param cross_validation_folds: Number of CV folds (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys outcome_column, n_training, n_test,
        cross_validated_r2, test_r2, test_rmse, test_mae, optimal_alpha.
    """
    covariate_columns = list(demographic_covariate_columns)

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=outcome_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        stratification_column=stratification_column,
    )

    ridge_pipeline = Pipeline(steps=[
        ("preprocess", build_activity_scaling_preprocessor(
            demographic_covariate_columns=covariate_columns,
            activity_predictor_columns=activity_predictor_columns,
        )),
        ("model", RidgeCV(
            alphas=np.logspace(-3, 3, 25),
            cv=cross_validation_folds,
        )),
    ])

    # Cross-validated R² on the training set — the scaler refits inside
    # each fold, so validation rows never influence the scaling.
    cross_validation_scores = cross_val_score(
        estimator=ridge_pipeline,
        X=training_predictors,
        y=training_outcome,
        cv=KFold(
            n_splits=cross_validation_folds,
            shuffle=True,
            random_state=random_state,
        ),
        scoring="r2",
    )

    # Fit once on the full training set for the held-out test evaluation.
    ridge_pipeline.fit(training_predictors, training_outcome)
    test_predictions = ridge_pipeline.predict(test_predictors)

    test_r2 = r2_score(test_outcome, test_predictions)
    test_rmse = float(np.sqrt(mean_squared_error(test_outcome, test_predictions)))
    test_mae = float(mean_absolute_error(test_outcome, test_predictions))
    optimal_alpha = float(ridge_pipeline.named_steps["model"].alpha_)

    print(
        f"\n  Cross-validated R² (train, 10-fold) : "
        f"{cross_validation_scores.mean():.4f}\n"
        f"  Test R²                             : {test_r2:.4f}\n"
        f"  Test RMSE                           : {test_rmse:.4f}\n"
        f"  Test MAE                            : {test_mae:.4f}\n"
        f"  Optimal alpha                       : {optimal_alpha:.6f}"
    )

    return {
        "outcome_column": outcome_column,
        "n_training": len(training_predictors),
        "n_test": len(test_predictors),
        "cross_validated_r2": float(cross_validation_scores.mean()),
        "test_r2": test_r2,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "optimal_alpha": optimal_alpha,
    }


#### Execute Ridge for all outcomes

In [ ]:
ridge_results: dict[str, list[dict]] = {}

for group_name, outcome_columns in OUTCOME_GROUPS.items():
    print(f"\n{'=' * 70}")
    print(f"Group: {group_name}  ({len(outcome_columns)} outcomes)")
    print(f"{'=' * 70}")

    group_results = []

    for outcome_column in outcome_columns:
        print(f"\n  Outcome: {outcome_column}")
        print(f"  {'-' * 50}")

        try:
            result = run_ridge_for_outcome(
                outcome_column=outcome_column,
                dataframe=summary_metrics_06,
                activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
                demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
                stratification_column=STRUCTURAL_COVARIATE_COLUMN,
                cross_validation_folds=CROSS_VALIDATION_FOLDS,
                random_state=RANDOM_STATE,
            )
            group_results.append(result)

        except ValueError as error:
            print(f"  Skipped: {error}")

    ridge_results[group_name] = group_results

#### Ridge summary table

In [ ]:
summary_rows = []

for group_name, group_results in ridge_results.items():
    for result in group_results:
        summary_rows.append({
            "group": group_name,
            "outcome": result["outcome_column"],
            "n_training": result["n_training"],
            "cv_r2_train": round(result["cross_validated_r2"], 4),
            "test_r2": round(result["test_r2"], 4),
            "test_rmse": round(result["test_rmse"], 4),
            "test_mae": round(result["test_mae"], 4),
            "optimal_alpha": round(result["optimal_alpha"], 6),
        })

ridge_summary_table = pd.DataFrame(summary_rows)

print("\nStage 2 — Ridge summary (sorted by test R²):")
print(
    ridge_summary_table
    .sort_values("test_r2", ascending=False)
    .to_string(index=False)
)

ridge_summary_table.to_csv(
    prediction_output / "stage_2_ridge_summary.csv",
    index=False,
)

### 1.2.3 Elastic Net

**Purpose.**
Elastic Net is the second predictive model in the pipeline and serves
as the bridge between LASSO (Stage 1) and Ridge (Stage 2). It combines
L1 and L2 regularisation in a single penalty term controlled by two
hyperparameters: alpha (overall regularisation strength) and l1_ratio
(the balance between L1 and L2).



**Hyperparameters.**
Both alpha and l1_ratio are tuned jointly via ElasticNetCV using
10-fold cross-validation on the training set. The l1_ratio grid
covers the full spectrum from predominantly Ridge (0.1) to pure
LASSO (1.0), with denser coverage in the sparser region:
[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0]. This allows the model to
select pure LASSO behaviour when sparsity is optimal, or a blend
when correlated predictors benefit from joint shrinkage.

Activity predictors are standardised on the training set; the same
scaler is applied to the test set. Covariates are passed through
unscaled, consistent with Stages 1 and 2.

In [ ]:
def run_elastic_net_for_outcome(
    *,
    outcome_column: str,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    stratification_column: str,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit an Elastic Net model with jointly tuned alpha and l1_ratio for
    a single outcome and return held-out test set metrics.

    Alpha and l1_ratio are selected jointly via ElasticNetCV using
    10-fold cross-validation on the training set. The l1_ratio grid
    covers [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0], allowing the model
    to select anywhere from predominantly Ridge to pure LASSO behaviour.

    Activity predictors are standardised within a pipeline so the scaler
    refits inside each cross-validation fold; validation rows never
    influence the scaling. Covariates pass through unscaled, consistent
    with Stages 1 and 2. The fitted pipeline applies the training-derived
    scaling to the held-out test set without refitting.

    :param outcome_column: Name of the outcome variable to model.
    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names
        (standardised within the pipeline).
    :param demographic_covariate_columns: Demographic covariate column
        names (passed through unscaled).
    :param stratification_column: KL grade column used to stratify the
        train/test split; not returned as a predictor.
    :param cross_validation_folds: Number of CV folds (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys outcome_column, n_training, n_test,
        cross_validated_r2, test_r2, test_rmse, test_mae,
        optimal_alpha, optimal_l1_ratio.
    """
    covariate_columns = list(demographic_covariate_columns)

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=outcome_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        stratification_column=stratification_column,
    )

    elastic_net_pipeline = Pipeline(steps=[
        ("preprocess", build_activity_scaling_preprocessor(
            demographic_covariate_columns=covariate_columns,
            activity_predictor_columns=activity_predictor_columns,
        )),
        ("model", ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
            cv=cross_validation_folds,
            random_state=random_state,
            max_iter=50_000,
        )),
    ])

    # Cross-validated R² on the training set — the scaler refits inside
    # each fold, so validation rows never influence the scaling.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cross_validation_scores = cross_val_score(
            estimator=elastic_net_pipeline,
            X=training_predictors,
            y=training_outcome,
            cv=KFold(
                n_splits=cross_validation_folds,
                shuffle=True,
                random_state=random_state,
            ),
            scoring="r2",
        )

    # Fit once on the full training set for the held-out test evaluation.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        elastic_net_pipeline.fit(training_predictors, training_outcome)

    test_predictions = elastic_net_pipeline.predict(test_predictors)
    test_r2 = r2_score(test_outcome, test_predictions)
    test_rmse = float(np.sqrt(mean_squared_error(test_outcome, test_predictions)))
    test_mae = float(mean_absolute_error(test_outcome, test_predictions))

    fitted_elastic_net = elastic_net_pipeline.named_steps["model"]
    optimal_alpha = float(fitted_elastic_net.alpha_)
    optimal_l1_ratio = float(fitted_elastic_net.l1_ratio_)

    # build_activity_scaling_preprocessor lists the covariate passthrough
    # first and the activity block second, so the fitted coefficient vector
    # is ordered [covariates..., activity...]. Same slice assumption as the
    # LASSO stage; keep the two in sync.
    number_of_covariates = len(covariate_columns)
    activity_coefficients = fitted_elastic_net.coef_[number_of_covariates:]
    non_zero_mask = activity_coefficients != 0

    selected_predictors = [
        column
        for column, is_selected in zip(activity_predictor_columns, non_zero_mask)
        if is_selected
    ]

    elastic_net_coefficients = pd.Series(
        data=activity_coefficients[non_zero_mask],
        index=selected_predictors,
        name="elastic_net_coefficient",
    ).sort_values(key=abs, ascending=False)

    print(
        f"\n  Cross-validated R² (train, 10-fold) : "
        f"{cross_validation_scores.mean():.4f}\n"
        f"  Test R²                             : {test_r2:.4f}\n"
        f"  Test RMSE                           : {test_rmse:.4f}\n"
        f"  Test MAE                            : {test_mae:.4f}\n"
        f"  Optimal alpha                       : {optimal_alpha:.6f}\n"
        f"  Optimal l1_ratio                    : {optimal_l1_ratio:.2f}"
    )

    return {
        "outcome_column": outcome_column,
        "n_training": len(training_predictors),
        "n_test": len(test_predictors),
        "cross_validated_r2": float(cross_validation_scores.mean()),
        "test_r2": test_r2,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "optimal_alpha": optimal_alpha,
        "optimal_l1_ratio": optimal_l1_ratio,
        "selected_predictors": selected_predictors,
        "elastic_net_coefficients": elastic_net_coefficients,
        "n_selected_predictors": len(selected_predictors),
    }

#### Execute Elastic Net for all outcomes

In [ ]:
elastic_net_results: dict[str, list[dict]] = {}

for group_name, outcome_columns in OUTCOME_GROUPS.items():
    print(f"\n{'=' * 70}")
    print(f"Group: {group_name}  ({len(outcome_columns)} outcomes)")
    print(f"{'=' * 70}")

    group_results = []

    for outcome_column in outcome_columns:
        print(f"\n  Outcome: {outcome_column}")
        print(f"  {'-' * 50}")

        try:
            result = run_elastic_net_for_outcome(
                outcome_column=outcome_column,
                dataframe=summary_metrics_06,
                activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
                demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
                stratification_column=STRUCTURAL_COVARIATE_COLUMN,
                cross_validation_folds=CROSS_VALIDATION_FOLDS,
                random_state=RANDOM_STATE,
            )
            group_results.append(result)

        except ValueError as error:
            print(f"  Skipped: {error}")

    elastic_net_results[group_name] = group_results


#### Elastic Net summary table

In [ ]:
summary_rows = []

for group_name, group_results in elastic_net_results.items():
    for result in group_results:
        summary_rows.append({
            "group": group_name,
            "outcome": result["outcome_column"],
            "n_training": result["n_training"],
            "cv_r2_train": round(result["cross_validated_r2"], 4),
            "test_r2": round(result["test_r2"], 4),
            "test_rmse": round(result["test_rmse"], 4),
            "test_mae": round(result["test_mae"], 4),
            "optimal_alpha": round(result["optimal_alpha"], 6),
            "optimal_l1_ratio": round(result["optimal_l1_ratio"], 2),
            "n_selected_predictors": result["n_selected_predictors"],
        })

elastic_net_summary_table = pd.DataFrame(summary_rows)

print("\nStage 3 — Elastic Net summary (sorted by test R²):")
print(
    elastic_net_summary_table
    .sort_values("test_r2", ascending=False)
    .to_string(index=False)
)

elastic_net_summary_table.to_csv(
    prediction_output / "stage_3_elastic_net_summary.csv",
    index=False,
)

### 1.2.4 Random Forest

**Purpose.**
Random Forest is the first non-linear model in the pipeline. Unlike
the three linear models in Stages 1–3, Random Forest makes no
assumption about the functional form of the relationship between
predictors and outcomes. It can capture interactions between
predictors and non-linear effects that linear regularisation cannot
represent, making it a useful benchmark for whether the activity-
outcome relationships in this dataset are adequately described by
linear models.

**Hyperparameters.**
A single Random Forest regressor is fitted per outcome with the
following fixed hyperparameters:

- n_estimators = 500: enough trees for stable importance estimates
  at this sample size without excessive computation time
- max_depth = None: trees grow until leaves are pure or contain
  fewer than min_samples_leaf samples, allowing the model to
  capture deep interactions if they exist
- min_samples_leaf = 10: minimum 10 participants per leaf node,
  chosen to prevent overfitting on a dataset of approximately
  1,300–1,400 training participants
- No standardisation is applied: tree-based models are scale-
  invariant and do not require feature scaling

Hyperparameters are not grid-searched here. The fixed specification
is a deliberate choice for a dataset of this size — grid search over
Random Forest hyperparameters is computationally expensive and the
chosen defaults are well-validated for n ≈ 1,000–2,000 in the
literature.

In [ ]:
def run_random_forest_for_outcome(
    *,
    outcome_column: str,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    stratification_column: str,
    number_of_trees: int = 500,
    minimum_samples_per_leaf: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit a Random Forest regressor for a single outcome and return
    held-out test set metrics and feature importances.

    No standardisation is applied since tree-based models are scale-
    invariant. Hyperparameters are fixed rather than grid-searched:
    500 trees and a minimum of 10 samples per leaf are well-validated
    defaults for datasets of approximately 1,000–2,000 participants.

    Feature importances are mean decrease in impurity (MDI), reported
    for all predictors ranked by importance descending. MDI is biased
    toward high-cardinality continuous predictors; rankings should be
    read as indicative rather than definitive.

    :param outcome_column: Name of the outcome variable to model.
    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names.
    :param demographic_covariate_columns: Demographic covariate column
        names.
    :param stratification_column: KL grade column used to stratify the
        train/test split; not returned as a predictor.
    :param number_of_trees: Number of trees in the forest (default 500).
    :param minimum_samples_per_leaf: Minimum number of participants per
        leaf node (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys outcome_column, n_training, n_test,
        test_r2, test_rmse, test_mae, feature_importances.
    """
    covariate_columns = list(demographic_covariate_columns)

    all_predictor_columns = covariate_columns + activity_predictor_columns

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=outcome_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        stratification_column=stratification_column,
    )

    training_matrix = training_predictors[all_predictor_columns].values
    test_matrix = test_predictors[all_predictor_columns].values

    random_forest_model = RandomForestRegressor(
        n_estimators=number_of_trees,
        max_depth=None,
        min_samples_leaf=minimum_samples_per_leaf,
        n_jobs=-1,
        random_state=random_state,
    )
    random_forest_model.fit(training_matrix, training_outcome)

    test_predictions = random_forest_model.predict(test_matrix)
    test_r2 = r2_score(test_outcome, test_predictions)
    test_rmse = float(np.sqrt(mean_squared_error(test_outcome, test_predictions)))
    test_mae = float(mean_absolute_error(test_outcome, test_predictions))

    feature_importances = pd.Series(
        data=random_forest_model.feature_importances_,
        index=all_predictor_columns,
        name="importance",
    ).sort_values(ascending=False)

    print(
        f"\n  Test R²   : {test_r2:.4f}\n"
        f"  Test RMSE : {test_rmse:.4f}\n"
        f"  Test MAE  : {test_mae:.4f}\n"
        f"\n  Top 10 feature importances (MDI):"
    )
    for predictor_name, importance_value in feature_importances.head(10).items():
        print(f"    {predictor_name:<45} {importance_value:.4f}")

    return {
        "outcome_column": outcome_column,
        "n_training": len(training_predictors),
        "n_test": len(test_predictors),
        "test_r2": test_r2,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "feature_importances": feature_importances,
    }

#### Execute Random Forest for all outcomes

In [ ]:
random_forest_results: dict[str, list[dict]] = {}

for group_name, outcome_columns in OUTCOME_GROUPS.items():
    print(f"\n{'=' * 70}")
    print(f"Group: {group_name}  ({len(outcome_columns)} outcomes)")
    print(f"{'=' * 70}")

    group_results = []

    for outcome_column in outcome_columns:
        print(f"\n  Outcome: {outcome_column}")
        print(f"  {'-' * 50}")

        try:
            result = run_random_forest_for_outcome(
                outcome_column=outcome_column,
                dataframe=summary_metrics_06,
                activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
                demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
                stratification_column=STRUCTURAL_COVARIATE_COLUMN,
                number_of_trees=500,
                minimum_samples_per_leaf=10,
                random_state=RANDOM_STATE,
            )
            group_results.append(result)

        except ValueError as error:
            print(f"  Skipped: {error}")

    random_forest_results[group_name] = group_results


#### Random Forest summary table

In [ ]:
summary_rows = []

for group_name, group_results in random_forest_results.items():
    for result in group_results:
        summary_rows.append({
            "group": group_name,
            "outcome": result["outcome_column"],
            "n_training": result["n_training"],
            "test_r2": round(result["test_r2"], 4),
            "test_rmse": round(result["test_rmse"], 4),
            "test_mae": round(result["test_mae"], 4),
        })

random_forest_summary_table = pd.DataFrame(summary_rows)

print("\nStage 4 — Random Forest summary (sorted by test R²):")
print(
    random_forest_summary_table
    .sort_values("test_r2", ascending=False)
    .to_string(index=False)
)

random_forest_summary_table.to_csv(
    prediction_output / "stage_4_random_forest_summary.csv",
    index=False,
)

### 1.2.5 XGBoost

**Purpose.**
XGBoost is the final and most flexible model in the pipeline. It fits
an ensemble of gradient-boosted decision trees sequentially, where
each tree corrects the residual errors of the previous ensemble.
Compared to Random Forest (Stage 4), which builds trees independently
in parallel, XGBoost uses a more targeted learning process that often
achieves higher predictive accuracy at the cost of additional
hyperparameters requiring tuning.

**Hyperparameters.**
A focused grid search is performed over two hyperparameters:

- max_depth: [3, 4, 6] — controls tree complexity and the depth
  of interactions captured. Shallow trees (3) produce smoother,
  more regularised fits; deeper trees (6) can capture higher-order
  interactions at the risk of overfitting.
- learning_rate: [0.03, 0.05, 0.1] — controls the contribution
  of each tree to the ensemble. Lower rates require more trees but
  generalise better.

All other hyperparameters are fixed:
- n_estimators = 500: sufficient trees for convergence at the
  learning rates used
- subsample = 0.8: each tree is fitted on 80% of the training
  data, reducing variance
- colsample_bytree = 0.8: each tree uses 80% of predictors,
  further reducing variance and computation time

Grid search uses 10-fold cross-validation on the training set with
R² as the scoring metric, yielding 9 configurations × 10 folds =
90 fits per outcome. No standardisation is applied since tree-based
models are scale-invariant.

In [ ]:
def run_xgboost_for_outcome(
    *,
    outcome_column: str,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    stratification_column: str,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit an XGBoost gradient-boosted regressor with grid-searched
    hyperparameters for a single outcome and return held-out test
    set metrics and feature importances.

    A focused grid search over max_depth [3, 4, 6] and learning_rate
    [0.03, 0.05, 0.1] is performed using 10-fold cross-validation on
    the training set. All other hyperparameters are fixed: 500 trees,
    subsample 0.8, colsample_bytree 0.8.

    No standardisation is applied since tree-based models are scale-
    invariant. Feature importances are gain-based, measuring the
    average loss improvement per split where each feature is used.

    :param outcome_column: Name of the outcome variable to model.
    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names.
    :param demographic_covariate_columns: Demographic covariate column
        names.
    :param stratification_column: KL grade column used to stratify the
        train/test split; not returned as a predictor.
    :param cross_validation_folds: Number of CV folds (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys outcome_column, n_training, n_test,
        cross_validated_r2, test_r2, test_rmse, test_mae,
        best_hyperparameters, feature_importances.
    """
    covariate_columns = list(demographic_covariate_columns)

    all_predictor_columns = covariate_columns + activity_predictor_columns

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=outcome_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        stratification_column=stratification_column,
    )

    training_matrix = training_predictors[all_predictor_columns].values
    test_matrix = test_predictors[all_predictor_columns].values

    hyperparameter_grid = {
        "max_depth": [3, 4, 6],
        "learning_rate": [0.03, 0.05, 0.1],
    }

    grid_search = GridSearchCV(
        estimator=xgboost.XGBRegressor(
            n_estimators=500,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        ),
        param_grid=hyperparameter_grid,
        cv=KFold(
            n_splits=cross_validation_folds,
            shuffle=True,
            random_state=random_state,
        ),
        scoring="r2",
        n_jobs=-1,
    )
    grid_search.fit(training_matrix, training_outcome)

    best_model = grid_search.best_estimator_
    test_predictions = best_model.predict(test_matrix)
    test_r2 = r2_score(test_outcome, test_predictions)
    test_rmse = float(np.sqrt(mean_squared_error(test_outcome, test_predictions)))
    test_mae = float(mean_absolute_error(test_outcome, test_predictions))

    feature_importances = pd.Series(
        data=best_model.feature_importances_,
        index=all_predictor_columns,
        name="importance",
    ).sort_values(ascending=False)

    print(
        f"\n  Cross-validated R² (train, 10-fold) : "
        f"{grid_search.best_score_:.4f}\n"
        f"  Test R²                             : {test_r2:.4f}\n"
        f"  Test RMSE                           : {test_rmse:.4f}\n"
        f"  Test MAE                            : {test_mae:.4f}\n"
        f"  Best hyperparameters                : "
        f"{grid_search.best_params_}\n"
        f"\n  Top 10 feature importances (gain):"
    )
    for predictor_name, importance_value in feature_importances.head(10).items():
        print(f"    {predictor_name:<45} {importance_value:.4f}")

    return {
        "outcome_column": outcome_column,
        "n_training": len(training_predictors),
        "n_test": len(test_predictors),
        "cross_validated_r2": float(grid_search.best_score_),
        "test_r2": test_r2,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "best_hyperparameters": grid_search.best_params_,
        "feature_importances": feature_importances,
    }

#### Execute XGBoost for all outcomes

In [ ]:
xgboost_results: dict[str, list[dict]] = {}

for group_name, outcome_columns in OUTCOME_GROUPS.items():
    print(f"\n{'=' * 70}")
    print(f"Group: {group_name}  ({len(outcome_columns)} outcomes)")
    print(f"{'=' * 70}")

    group_results = []

    for outcome_column in outcome_columns:
        print(f"\n  Outcome: {outcome_column}")
        print(f"  {'-' * 50}")

        try:
            result = run_xgboost_for_outcome(
                outcome_column=outcome_column,
                dataframe=summary_metrics_06,
                activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
                demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
                stratification_column=STRUCTURAL_COVARIATE_COLUMN,
                cross_validation_folds=CROSS_VALIDATION_FOLDS,
                random_state=RANDOM_STATE,
            )
            group_results.append(result)

        except ValueError as error:
            print(f"  Skipped: {error}")

    xgboost_results[group_name] = group_results

#### XGBoost summary table

In [ ]:
summary_rows = []

for group_name, group_results in xgboost_results.items():
    for result in group_results:
        summary_rows.append({
            "group": group_name,
            "outcome": result["outcome_column"],
            "n_training": result["n_training"],
            "cv_r2_train": round(result["cross_validated_r2"], 4),
            "test_r2": round(result["test_r2"], 4),
            "test_rmse": round(result["test_rmse"], 4),
            "test_mae": round(result["test_mae"], 4),
            "max_depth": result["best_hyperparameters"]["max_depth"],
            "learning_rate": result["best_hyperparameters"]["learning_rate"],
        })

xgboost_summary_table = pd.DataFrame(summary_rows)

print("\nStage 5 — XGBoost summary (sorted by test R²):")
print(
    xgboost_summary_table
    .sort_values("test_r2", ascending=False)
    .to_string(index=False)
)

xgboost_summary_table.to_csv(
    prediction_output / "stage_5_xgboost_summary.csv",
    index=False,
)

### 1.2.6 KL grade classification

**Purpose.**
Stage 6 treats KL grade as a multiclass ordinal outcome rather than
a covariate. The question being asked is different from Stages 1–5:
not "how well do activity features predict a clinical symptom outcome"
but "how much does the activity pattern alone reveal about the
structural severity of a participant's knee osteoarthritis".

**Model specification.**

Two classifiers are fitted:

Random Forest classifier:
- 500 trees, balanced class weights, min 10 samples per leaf
- No standardisation required

XGBoost classifier:
- 400 trees, max_depth 4, learning_rate 0.05
- subsample 0.8, colsample_bytree 0.8
- Inverse frequency sample weights for class imbalance
- Hyperparameters are fixed rather than grid-searched to keep
  runtime manageable; the specification follows the same
  rationale as Stage 5

#### Class-balanced XGBoost wrapper

In [ ]:
class InverseFrequencyWeightedXGBClassifier(xgboost.XGBClassifier):
    """
    XGBoost classifier that applies balanced inverse-frequency sample
    weights computed from the training rows it is fitted on.

    Computing the weights inside ``fit`` (rather than precomputing them
    and passing them through the ``sample_weight`` argument) means the
    weighting survives estimator cloning and is recomputed on each
    cross-validation fold's own class distribution. This keeps the
    cross-validated metric and the final held-out fit on an identical
    weighting procedure.

    ``compute_sample_weight(class_weight="balanced")`` implements the
    same n_samples / (n_classes * bincount) inverse-frequency scheme,
    normalised so the weights sum to the sample count, which avoids
    over-correction and stops the classifier collapsing to a single
    predicted class.
    """

    def fit(self, X, y, **kwargs):
        """
        Fit with balanced inverse-frequency sample weights from y.

        :param X: Training feature matrix.
        :param y: Training class labels.
        :param kwargs: Additional arguments forwarded to
            ``xgboost.XGBClassifier.fit``.
        :returns: The fitted estimator.
        """
        sample_weight = compute_sample_weight(class_weight="balanced", y=y)
        return super().fit(X, y, sample_weight=sample_weight, **kwargs)

In [ ]:
def run_kl_grade_classification(
    *,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    structural_covariate_column: str,
    model_name: str,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit a multiclass classifier for KL grade (0 to 4) and evaluate
    using metrics appropriate for an ordinal outcome.

    KL grade is the outcome here and is never included as a predictor.
    The train/test split is stratified by KL grade itself. Class
    imbalance is handled via balanced class weights for Random Forest
    and normalised inverse frequency sample weights for XGBoost.

    Quadratic weighted kappa is the primary evaluation metric. It
    penalises predictions in proportion to the squared distance from
    the true class, which is appropriate for an ordinal outcome where
    distant misclassifications are more serious than adjacent ones.

    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names.
    :param demographic_covariate_columns: Demographic covariate column
        names. Must not contain KL grade.
    :param structural_covariate_column: KL grade column name. Used as
        the outcome and as the stratification key — not included as a
        predictor.
    :param model_name: Either ``"random_forest"`` or ``"xgboost"``.
    :param cross_validation_folds: Number of CV folds (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys cross_validated_kappa, test_kappa,
        test_accuracy, test_mean_absolute_error_grade,
        feature_importances, confusion_matrix.
    :raises ValueError: If ``model_name`` is not recognised or if KL
        grade appears in ``demographic_covariate_columns``.
    """
    if structural_covariate_column in demographic_covariate_columns:
        raise ValueError(
            f"KL grade column '{structural_covariate_column}' must not "
            f"appear in demographic_covariate_columns when modelling KL "
            f"grade as the outcome — this would cause data leakage."
        )

    if model_name not in {"random_forest", "xgboost"}:
        raise ValueError(
            f"Unrecognised model_name '{model_name}'. "
            f"Expected 'random_forest' or 'xgboost'."
        )

    all_predictor_columns = (
        list(demographic_covariate_columns) + activity_predictor_columns
    )

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=structural_covariate_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        stratification_column=structural_covariate_column,
    )

    training_matrix = training_predictors[all_predictor_columns].values
    test_matrix = test_predictors[all_predictor_columns].values

    training_classes = training_outcome.astype(int).values
    test_classes = test_outcome.astype(int).values

    if model_name == "random_forest":
        classifier = RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=10,
            class_weight="balanced",
            n_jobs=-1,
            random_state=random_state,
        )
        classifier.fit(training_matrix, training_classes)

    else:
        classifier = InverseFrequencyWeightedXGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softmax",
            num_class=int(np.max(training_classes)) + 1,
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        )
        classifier.fit(training_matrix, training_classes)

    def score_quadratic_weighted_kappa(
        estimator,
        predictor_matrix,
        true_values,
    ) -> float:
        """
        Scorer function for cross_val_score computing quadratic weighted
        kappa between true and predicted KL grade classes.

        :param estimator: Fitted classifier with a predict method.
        :param predictor_matrix: Feature matrix to predict from.
        :param true_values: True integer-coded KL grade values.
        :returns: Quadratic weighted kappa score.
        """
        return cohen_kappa_score(
            y1=true_values,
            y2=estimator.predict(predictor_matrix),
            weights="quadratic",
        )

    cross_validation_kappa_scores = cross_val_score(
        estimator=classifier.__class__(**classifier.get_params()),
        X=training_matrix,
        y=training_classes,
        cv=StratifiedKFold(
            n_splits=cross_validation_folds,
            shuffle=True,
            random_state=random_state,
        ),
        scoring=score_quadratic_weighted_kappa,
        n_jobs=-1,
    )

    test_predictions = classifier.predict(test_matrix)

    test_kappa = cohen_kappa_score(
        y1=test_classes,
        y2=test_predictions,
        weights="quadratic",
    )
    test_accuracy = accuracy_score(test_classes, test_predictions)
    test_mean_absolute_error_grade = float(
        np.mean(np.abs(test_classes - test_predictions))
    )

    feature_importances = pd.Series(
        data=classifier.feature_importances_,
        index=all_predictor_columns,
        name="importance",
    ).sort_values(ascending=False)

    confusion_matrix = pd.crosstab(
        pd.Series(test_classes, name="actual"),
        pd.Series(test_predictions, name="predicted"),
    )

    print(
        f"\n  Cross-validated kappa (train, {cross_validation_folds}-fold) : "
        f"{cross_validation_kappa_scores.mean():.4f}\n"
        f"  Test quadratic weighted kappa          : {test_kappa:.4f}\n"
        f"  Test accuracy                          : {test_accuracy:.4f}\n"
        f"  Test MAE (grade)                       : "
        f"{test_mean_absolute_error_grade:.4f}\n"
        f"\n  Confusion matrix (rows: actual, columns: predicted):\n"
        f"{confusion_matrix}\n"
        f"\n  Top 10 feature importances:"
    )
    for predictor_name, importance_value in feature_importances.head(10).items():
        print(f"    {predictor_name:<45} {importance_value:.4f}")

    return {
        "cross_validated_kappa": float(cross_validation_kappa_scores.mean()),
        "test_kappa": test_kappa,
        "test_accuracy": test_accuracy,
        "test_mean_absolute_error_grade": test_mean_absolute_error_grade,
        "feature_importances": feature_importances,
        "confusion_matrix": confusion_matrix,
    }

#### Execute KL grade classification for both models

In [ ]:
print(f"\n{'=' * 70}")
print("Stage 6 — KL grade classification")
print(f"{'=' * 70}")

kl_grade_results: dict[str, dict] = {}

for model_name in ["random_forest", "xgboost"]:
    print(f"\n  Model: {model_name}")
    print(f"  {'-' * 50}")

    kl_grade_results[model_name] = run_kl_grade_classification(
        dataframe=summary_metrics_06,
        activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
        demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
        structural_covariate_column=STRUCTURAL_COVARIATE_COLUMN,
        model_name=model_name,
        cross_validation_folds=CROSS_VALIDATION_FOLDS,
        random_state=RANDOM_STATE,
    )

#### KL grade classification summary table

In [ ]:
print(f"\n{'=' * 70}")
print("Stage 6 — KL grade classification summary")
print(f"{'=' * 70}")

for model_name, result in kl_grade_results.items():
    print(
        f"\n  {model_name.replace('_', ' ').title()}\n"
        f"    CV kappa (train)          : "
        f"{result['cross_validated_kappa']:.4f}\n"
        f"    Test kappa (QWK)          : {result['test_kappa']:.4f}\n"
        f"    Test accuracy             : {result['test_accuracy']:.4f}\n"
        f"    Test MAE (grade)          : "
        f"{result['test_mean_absolute_error_grade']:.4f}"
    )

best_model_name = max(
    kl_grade_results,
    key=lambda name: kl_grade_results[name]["test_kappa"],
)
print(
    f"\n  Best model: {best_model_name.replace('_', ' ').title()} "
    f"(test kappa = "
    f"{kl_grade_results[best_model_name]['test_kappa']:.4f})"
)

kl_grade_summary_table = pd.DataFrame(
    [
        {
            "model": model_name,
            "cross_validated_kappa": result["cross_validated_kappa"],
            "test_kappa": result["test_kappa"],
            "test_accuracy": result["test_accuracy"],
            "test_mean_absolute_error_grade": result["test_mean_absolute_error_grade"],
        }
        for model_name, result in kl_grade_results.items()
    ]
)
kl_grade_summary_table.to_csv(
    prediction_output / "stage_6_kl_grade_classification.csv",
    index=False,
)

## 1.3 Cross-model comparison

**Purpose.**

Stage 7 assembles the results from all previous stages into a single
comparison table per outcome, enabling direct comparison of all five
model families on a common held-out test set metric. This is the
primary results table for the thesis modelling chapter.

**What is being compared.**
For continuous outcomes (Stages 1–5), the comparison metric is test
R² — the proportion of variance in the held-out test set explained
by each model. All models were evaluated on the same 20% held-out
test set produced by the same stratified split, so test R² values
are directly comparable across model families.

Stage 1 LASSO contributes full_r2_cv rather than a test set R²
because its role is explanatory rather than predictive. It is included
in the table for reference but is not directly comparable to the
held-out test R² values from Stages 2–5. The incremental_r2 from
Stage 1 is reported as a separate column.

For KL grade (Stage 6), the comparison metric is quadratic weighted
kappa. This is reported in a separate table.

**Best model selection.**
The best model per outcome is selected as the model family with the
highest test R² among Stages 2–5. Stage 1 is excluded from best
model selection because it optimises for explanation rather than
prediction.

In [ ]:
def build_continuous_outcome_comparison_table(
    *,
    lasso_results: dict[str, list[dict]],
    ridge_results: dict[str, list[dict]],
    elastic_net_results: dict[str, list[dict]],
    random_forest_results: dict[str, list[dict]],
    xgboost_results: dict[str, list[dict]],
) -> pd.DataFrame:
    """
    Assemble a single comparison table across all five model families
    for every continuous outcome modelled in Stages 1–5.

    For each outcome the table contains:
    - Stage 1 LASSO: incremental_r2 and full_r2_cv (explanatory,
      training set CV — not directly comparable to test R²)
    - Stages 2–5: test_r2 on the held-out test set
    - best_model: model family with the highest test R² among
      Stages 2–5
    - best_test_r2: the corresponding test R²

    :param lasso_results: Output of Stage 1 execution cell.
    :param ridge_results: Output of Stage 2 execution cell.
    :param elastic_net_results: Output of Stage 3 execution cell.
    :param random_forest_results: Output of Stage 4 execution cell.
    :param xgboost_results: Output of Stage 5 execution cell.
    :returns: DataFrame with one row per outcome sorted by
        best_test_r2 descending.
    """
    lasso_lookup: dict[str, dict] = {
        str(result["outcome_column"]): result
        for group_results in lasso_results.values()
        for result in group_results
    }
    ridge_lookup: dict[str, dict] = {
        str(result["outcome_column"]): result
        for group_results in ridge_results.values()
        for result in group_results
    }
    elastic_net_lookup: dict[str, dict] = {
        str(result["outcome_column"]): result
        for group_results in elastic_net_results.values()
        for result in group_results
    }
    random_forest_lookup: dict[str, dict] = {
        str(result["outcome_column"]): result
        for group_results in random_forest_results.values()
        for result in group_results
    }
    xgboost_lookup: dict[str, dict] = {
        str(result["outcome_column"]): result
        for group_results in xgboost_results.values()
        for result in group_results
    }

    all_outcome_columns = list(ridge_lookup.keys())

    rows = []

    for outcome_column in all_outcome_columns:
        lasso_result = lasso_lookup.get(outcome_column, {})
        ridge_result = ridge_lookup.get(outcome_column, {})
        elastic_net_result = elastic_net_lookup.get(outcome_column, {})
        random_forest_result = random_forest_lookup.get(outcome_column, {})
        xgboost_result = xgboost_lookup.get(outcome_column, {})

        ridge_test_r2 = ridge_result.get("test_r2", np.nan)
        elastic_net_test_r2 = elastic_net_result.get("test_r2", np.nan)
        random_forest_test_r2 = random_forest_result.get("test_r2", np.nan)
        xgboost_test_r2 = xgboost_result.get("test_r2", np.nan)

        predictive_model_r2_values = {
            "ridge": ridge_test_r2,
            "elastic_net": elastic_net_test_r2,
            "random_forest": random_forest_test_r2,
            "xgboost": xgboost_test_r2,
        }

        best_model_name = max(
            predictive_model_r2_values,
            key=lambda name: (
                predictive_model_r2_values[name]
                if not np.isnan(predictive_model_r2_values[name])
                else -np.inf
            ),
        )
        best_test_r2 = predictive_model_r2_values[best_model_name]

        group_name = next(
            (
                group
                for group, group_results in ridge_results.items()
                for result in group_results
                if result["outcome_column"] == outcome_column
            ),
            "unknown",
        )

        rows.append({
            "group": group_name,
            "outcome": outcome_column,
            "n_training": ridge_result.get("n_training", np.nan),
            "lasso_incremental_r2": round(
                lasso_result.get("incremental_r2", np.nan), 4
            ),
            "lasso_full_r2_cv": round(
                lasso_result.get("full_r2_cv", np.nan), 4
            ),
            "ridge_test_r2": round(ridge_test_r2, 4),
            "elastic_net_test_r2": round(elastic_net_test_r2, 4),
            "random_forest_test_r2": round(random_forest_test_r2, 4),
            "xgboost_test_r2": round(xgboost_test_r2, 4),
            "best_model": best_model_name,
            "best_test_r2": round(best_test_r2, 4),
        })

    comparison_table = (
        pd.DataFrame(rows)
        .sort_values("best_test_r2", ascending=False)
        .reset_index(drop=True)
    )

    return comparison_table


def print_comparison_table_by_group(
    comparison_table: pd.DataFrame,
) -> None:
    """
    Print the comparison table grouped by outcome domain for
    readability in the notebook output.

    :param comparison_table: Output of
        build_continuous_outcome_comparison_table.
    """
    column_widths = {
        "outcome": 40,
        "n_training": 10,
        "lasso_incremental_r2": 14,
        "lasso_full_r2_cv": 14,
        "ridge_test_r2": 13,
        "elastic_net_test_r2": 16,
        "random_forest_test_r2": 18,
        "xgboost_test_r2": 13,
        "best_model": 14,
        "best_test_r2": 12,
    }

    header = (
        f"{'Outcome':<40} {'n_train':>10} {'LASSO_incR²':>14} "
        f"{'LASSO_cvR²':>14} {'Ridge_R²':>13} {'ElNet_R²':>16} "
        f"{'RF_R²':>18} {'XGB_R²':>13} {'Best':>14} {'BestR²':>12}"
    )
    separator = "-" * len(header)

    for group_name, group_data in comparison_table.groupby("group", sort=False):
        print(f"\n{'=' * len(header)}")
        print(f"Group: {group_name}")
        print(f"{'=' * len(header)}")
        print(header)
        print(separator)

        for _, row in group_data.iterrows():
            print(
                f"{row['outcome']:<40} "
                f"{row['n_training']:>10} "
                f"{row['lasso_incremental_r2']:>14.4f} "
                f"{row['lasso_full_r2_cv']:>14.4f} "
                f"{row['ridge_test_r2']:>13.4f} "
                f"{row['elastic_net_test_r2']:>16.4f} "
                f"{row['random_forest_test_r2']:>18.4f} "
                f"{row['xgboost_test_r2']:>13.4f} "
                f"{row['best_model']:>14} "
                f"{row['best_test_r2']:>12.4f}"
            )

### Execute comparison table construction and printing

In [ ]:
comparison_table = build_continuous_outcome_comparison_table(
    lasso_results=lasso_results,
    ridge_results=ridge_results,
    elastic_net_results=elastic_net_results,
    random_forest_results=random_forest_results,
    xgboost_results=xgboost_results,
)

print(f"\n{'=' * 70}")
print("Stage 7 — Cross-model comparison")
print(f"{'=' * 70}")

print_comparison_table_by_group(comparison_table=comparison_table)

comparison_table.to_csv(
    prediction_output / "stage_7_model_comparison.csv",
    index=False,
)

print(f"\nComparison table saved to: stage_7_model_comparison.csv")
print(f"Total outcomes compared: {len(comparison_table)}")

#### KL grade classification summary

In [ ]:
print(f"\n{'=' * 70}")
print("KL grade classification (Stage 6)")
print(f"{'=' * 70}")
print(
    f"\n{'Model':<20} {'CV kappa':>12} {'Test kappa':>12} "
    f"{'Accuracy':>12} {'MAE grade':>12}"
)
print("-" * 70)

for model_name, result in kl_grade_results.items():
    print(
        f"{model_name.replace('_', ' ').title():<20} "
        f"{result['cross_validated_kappa']:>12.4f} "
        f"{result['test_kappa']:>12.4f} "
        f"{result['test_accuracy']:>12.4f} "
        f"{result['test_mean_absolute_error_grade']:>12.4f}"
    )

# 2. Longitudinal validation (Visit 06 → Visit 08)

## 2.1 Outcome selection

Selects the outcomes carried into the longitudinal confirmatory tier. The set is fixed by pre-specification rather than by held-out performance: all modelled outcomes are carried forward except the 400m walk measures, which are assessed only at baseline and cannot be validated at Visit 08.

In [ ]:
EXCLUDED_OUTCOMES_400M = {"V06400MTIM", "V06400MTR"}

# The confirmatory outcome set is fixed by pre-specification: every
# modelled outcome is carried forward except the 400m walk measures,
# which are assessed only at baseline and cannot be validated
# longitudinally. No performance-based gate is applied, so the
# permutation test and its within-family FDR correction do not inherit a
# data-driven selection step. Absolute test R² is retained in the table
# for descriptive reporting, not as an inclusion criterion.
outcomes_after_exclusion = comparison_table[
    ~comparison_table["outcome"].isin(EXCLUDED_OUTCOMES_400M)
].copy()

print(
    f"Outcome selection for longitudinal validation\n"
    f"  Total outcomes modelled   : {len(comparison_table)}\n"
    f"  Excluded (400m walk)      : "
    f"{len(comparison_table) - len(outcomes_after_exclusion)}\n"
    f"  Carried to confirmatory   : {len(outcomes_after_exclusion)}"
)

## 2.2 Longitudinal model specification

For each selected outcome the winning model family (highest test R² among Ridge, Elastic Net, Random Forest, XGBoost) is refitted using the same hyperparameters on the full baseline dataset (no train/test split). Only that model will be re-fitted on the full Visit 06 training set and used to generate Visit 08 predictions during the longitudinal validation phase.

In [ ]:
LONGITUDINAL_VALIDATION_OUTCOMES: dict[str, dict] = {
    row["outcome"]: {
        "group": row["group"],
        "best_model": row["best_model"],
        "best_test_r2": row["best_test_r2"],
        "lasso_incremental_r2": row["lasso_incremental_r2"],
        "n_training": int(row["n_training"]),
    }
    for _, row in outcomes_after_exclusion.iterrows()
}

print(f"\nSelected outcomes ({len(LONGITUDINAL_VALIDATION_OUTCOMES)} total):\n")
print(
    f"  {'Outcome':<45} {'Group':<25} "
    f"{'Best model':<15} {'Test R²':>8} {'LASSO ΔR²':>10}"
)
print("  " + "-" * 105)

current_group = None

for outcome_column, metadata in sorted(
    LONGITUDINAL_VALIDATION_OUTCOMES.items(),
    key=lambda item: (item[1]["group"], -item[1]["best_test_r2"]),
):
    if metadata["group"] != current_group:
        current_group = metadata["group"]
        print(f"\n  [{current_group.upper()}]")

    print(
        f"  {outcome_column:<45} {metadata['group']:<25} "
        f"{metadata['best_model']:<15} "
        f"{metadata['best_test_r2']:>8.4f} "
        f"{metadata['lasso_incremental_r2']:>10.4f}"
    )


## 2.3 Refit winning models on full Visit 06

For each outcome selected for longitudinal validation, the winning model
family is refitted on the complete Visit 06 dataset (training and test
combined). The fitted model and its StandardScaler are stored in memory
for direct use in the Visit 08 prediction step that follows.

No train/test split is applied here. The held-out test set R² from
Stage 7 already provides the unbiased performance estimate.

In [ ]:
def refit_winning_model_on_full_visit_06(
        *,
        outcome_column: str,
        best_model: str,
        dataframe: pd.DataFrame,
        activity_predictor_columns: list[str],
        demographic_covariate_columns: list[str],
        ridge_results: dict[str, list[dict]],
        elastic_net_results: dict[str, list[dict]],
        random_forest_results: dict[str, list[dict]],
        xgboost_results: dict[str, list[dict]],
        random_state: int = 42,
) -> dict:
    """
    Refit the winning model family for a single outcome on the full
    Visit 06 dataset (no train/test split).

    The scaler is fitted on the full Visit 06 predictor matrix and
    stored alongside the model so that the same transformation can
    be applied to Visit 08 data without refitting.

    Hyperparameters are taken directly from the in-memory result
    dictionaries produced during Stages 2–5, so no cross-validation
    or grid search is repeated here.

    :param outcome_column: Name of the outcome variable to model.
    :param best_model: Winning model family name. One of ``"ridge"``,
        ``"elastic_net"``, ``"random_forest"``, or ``"xgboost"``.
    :param dataframe: Full Visit 06 modelling dataframe with one row
        per participant.
    :param activity_predictor_columns: Activity predictor column names.
    :param demographic_covariate_columns: Demographic covariate column
        names.
    :param ridge_results: In-memory Ridge results from Stage 2.
    :param elastic_net_results: In-memory Elastic Net results from Stage 3.
    :param random_forest_results: In-memory Random Forest results from Stage 4.
    :param xgboost_results: In-memory XGBoost results from Stage 5.
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys ``outcome_column``, ``best_model``,
        ``fitted_model``, ``scaler`` (or None for tree-based models),
        ``covariate_columns``, ``activity_predictor_columns``,
        and ``n_fitted``.
    :raises ValueError: If ``best_model`` is not a recognised model family.
    """
    # Training vintage is Visit 06 by design: the model learns the
    # activity-to-outcome mapping on the Visit 06 cross-section and is
    # then applied to an all-Visit-08 feature vector at evaluation time
    # (see resolve_predictor_to_visit_08). These covariates stay on V06.
    covariate_columns = list(demographic_covariate_columns)
    all_required_columns = covariate_columns + activity_predictor_columns + [outcome_column]

    complete_cases = dataframe[all_required_columns].dropna()
    number_of_complete_cases = len(complete_cases)

    print(
        f"  Refitting on full Visit 06 — {number_of_complete_cases:,} complete cases"
    )

    outcome_series = complete_cases[outcome_column]
    covariate_matrix = complete_cases[covariate_columns].values
    activity_matrix = complete_cases[activity_predictor_columns].values

    # Flatten result lists into outcome-keyed lookups
    ridge_lookup: dict[str, dict] = {
        str(result["outcome_column"]): result
        for group_results in ridge_results.values()
        for result in group_results
    }
    elastic_net_lookup: dict[str, dict] = {
        str(result["outcome_column"]): result
        for group_results in elastic_net_results.values()
        for result in group_results
    }
    xgboost_lookup: dict[str, dict] = {
        str(result["outcome_column"]): result
        for group_results in xgboost_results.values()
        for result in group_results
    }

    if best_model == "ridge":
        optimal_alpha = ridge_lookup[outcome_column]["optimal_alpha"]

        scaler = StandardScaler()
        activity_matrix_scaled = scaler.fit_transform(activity_matrix)
        full_matrix = np.hstack([covariate_matrix, activity_matrix_scaled])

        fitted_model = Ridge(alpha=optimal_alpha)
        fitted_model.fit(full_matrix, outcome_series)

        print(f"  Ridge — alpha={optimal_alpha:.6f}")

    elif best_model == "elastic_net":
        result = elastic_net_lookup[outcome_column]
        optimal_alpha = result["optimal_alpha"]
        optimal_l1_ratio = result["optimal_l1_ratio"]

        scaler = StandardScaler()
        activity_matrix_scaled = scaler.fit_transform(activity_matrix)
        full_matrix = np.hstack([covariate_matrix, activity_matrix_scaled])

        from sklearn.linear_model import ElasticNet
        fitted_model = ElasticNet(
            alpha=optimal_alpha,
            l1_ratio=optimal_l1_ratio,
            max_iter=50_000,
        )
        fitted_model.fit(full_matrix, outcome_series)

        print(
            f"  Elastic Net — alpha={optimal_alpha:.6f}, "
            f"l1_ratio={optimal_l1_ratio:.2f}"
        )

    elif best_model == "random_forest":
        scaler = None
        all_predictor_columns = covariate_columns + activity_predictor_columns
        full_matrix = complete_cases[all_predictor_columns].values

        fitted_model = RandomForestRegressor(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=10,
            n_jobs=-1,
            random_state=random_state,
        )
        fitted_model.fit(full_matrix, outcome_series)

        print("  Random Forest — n_estimators=500, min_samples_leaf=10")

    elif best_model == "xgboost":
        result = xgboost_lookup[outcome_column]
        best_hyperparameters = result["best_hyperparameters"]

        scaler = None
        all_predictor_columns = covariate_columns + activity_predictor_columns
        full_matrix = complete_cases[all_predictor_columns].values

        fitted_model = xgboost.XGBRegressor(
            n_estimators=500,
            max_depth=best_hyperparameters["max_depth"],
            learning_rate=best_hyperparameters["learning_rate"],
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        )
        fitted_model.fit(full_matrix, outcome_series)

        print(
            f"  XGBoost — max_depth={best_hyperparameters['max_depth']}, "
            f"learning_rate={best_hyperparameters['learning_rate']}"
        )

    else:
        raise ValueError(
            f"Unrecognised model family '{best_model}'. Expected one of: "
            f"ridge, elastic_net, random_forest, xgboost."
        )

    return {
        "outcome_column": outcome_column,
        "best_model": best_model,
        "fitted_model": fitted_model,
        "scaler": scaler,
        "covariate_columns": covariate_columns,
        "activity_predictor_columns": activity_predictor_columns,
        "n_fitted": number_of_complete_cases,
    }

In [ ]:
# Execute refit for all selected outcomes
refitted_models: dict[str, dict] = {}

for outcome_column, outcome_metadata in LONGITUDINAL_VALIDATION_OUTCOMES.items():
    print(f"\n{'=' * 60}")
    print(f"Outcome: {outcome_column}  [{outcome_metadata['best_model']}]")
    print(f"{'=' * 60}")

    refitted_models[outcome_column] = refit_winning_model_on_full_visit_06(
        outcome_column=outcome_column,
        best_model=outcome_metadata["best_model"],
        dataframe=summary_metrics_06,
        activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
        demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
        ridge_results=ridge_results,
        elastic_net_results=elastic_net_results,
        random_forest_results=random_forest_results,
        xgboost_results=xgboost_results,
        random_state=RANDOM_STATE,
    )

print(
    f"\nRefit complete — {len(refitted_models)} models ready for "
    f"Visit 08 prediction."
)

In [ ]:
# Execute the activity-only refit: same winning model family and
# hyperparameters per outcome as the combined refit, but with an empty
# covariate block. refit_winning_model_on_full_visit_06 handles
# demographic_covariate_columns=[] without modification -- the covariate
# matrix is zero-width and drops out of the horizontal stack.
#
# Caveat: best_model and its hyperparameters (alpha, l1_ratio, max_depth,
# ...) were selected in Stages 2-5 WITH covariates present. They are
# reused here rather than re-optimised for the covariate-free design -- a
# pragmatic simplification to note in the methods.
refitted_models_activity_only: dict[str, dict] = {}

for outcome_column, outcome_metadata in LONGITUDINAL_VALIDATION_OUTCOMES.items():
    print(f"\n{'=' * 60}")
    print(f"Outcome (activity-only): {outcome_column}  [{outcome_metadata['best_model']}]")
    print(f"{'=' * 60}")

    refitted_models_activity_only[outcome_column] = refit_winning_model_on_full_visit_06(
        outcome_column=outcome_column,
        best_model=outcome_metadata["best_model"],
        dataframe=summary_metrics_06,
        activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
        demographic_covariate_columns=[],
        ridge_results=ridge_results,
        elastic_net_results=elastic_net_results,
        random_forest_results=random_forest_results,
        xgboost_results=xgboost_results,
        random_state=RANDOM_STATE,
    )

print(
    f"\nActivity-only refit complete -- {len(refitted_models_activity_only)} models "
    f"ready for Visit 08 prediction."
)

## 2.4 Prepare Visit 08 dataset

Applies the same preparation to Visit 08 that was applied to Visit 06

In [ ]:
summary_metrics_08 = (
    summary_metrics_08.reset_index()
    if "ID" not in summary_metrics_08.columns
    else summary_metrics_08
)

# Defragment the frame in one pass so the column additions below don't
# trigger pandas' fragmentation PerformanceWarning.
summary_metrics_08 = summary_metrics_08.copy()

# Mirror the Visit 06 derivation exactly: derive a binary sex covariate
# from the raw P02SEX item. P02SEX is coded 1 = Male, 2 = Female in OAI;
# some exports carry string labels, so both encodings are mapped. The
# derived sex_female column uses 1 = female, 0 = male.
if "P02SEX" not in summary_metrics_08.columns:
    raise KeyError(
        "P02SEX is missing from summary_metrics_08. Inspect the available "
        "sex column with summary_metrics_08.filter(like='SEX').columns "
        "and adjust the derivation before running."
    )

sex_mapping = {
    1.0: 0.0,
    2.0: 1.0,
    "1": 0.0,
    "2": 1.0,
    "1: Male": 0.0,
    "2: Female": 1.0,
    "M": 0.0,
    "F": 1.0,
    "Male": 0.0,
    "Female": 1.0,
}

print("Raw P02SEX values (Visit 08):")
print(summary_metrics_08["P02SEX"].value_counts(dropna=False).to_string())

summary_metrics_08["sex_female"] = summary_metrics_08["P02SEX"].map(sex_mapping)

unmapped_sex_count = (
    summary_metrics_08["sex_female"].isna().sum()
    - summary_metrics_08["P02SEX"].isna().sum()
)
if unmapped_sex_count > 0:
    print(
        f"WARNING — {unmapped_sex_count} P02SEX values did not match the "
        f"mapping and became NaN. Inspect the raw values printed above and "
        f"extend sex_mapping before continuing."
    )

sex_missing = summary_metrics_08["sex_female"].isna().sum()
print(f"sex_female derived — missing in {sex_missing} participants")

print(
    f"Visit 08 dataset ready:\n"
    f"  Shape        : {summary_metrics_08.shape}\n"
    f"  Participants : {len(summary_metrics_08):,}"
)

## 2.5 Match Visit 06 and Visit 08 participants

Inner join on participant ID to retain only participants with complete
data at both visits. The Visit 06 observed outcome values are carried
forward as the baseline for direction-of-change computation.

In [ ]:
paired_dataframe = summary_metrics_06.merge(
    summary_metrics_08,
    on="ID",
    suffixes=("_v06", "_v08"),
)

print(
    f"Matched participants:\n"
    f"  Visit 06     : {len(summary_metrics_06):,}\n"
    f"  Visit 08     : {len(summary_metrics_08):,}\n"
    f"  Paired       : {len(paired_dataframe):,}"
)

## 2.6 Continuous outcome prediction and trajectory evaluation

In [ ]:
OUTCOME_HIGHER_IS_WORSE: dict[str, bool] = {
    # Pain
    "V06KOOSKPR":  False,  # KOOS pain: higher = better
    "V06KOOSKPL":  False,  # KOOS pain: higher = better
    "V06ICPTSKR":  True,   # ICOAP: higher = worse
    "V06ICPTSKL":  True,   # ICOAP: higher = worse

    # Function
    "V0620MPACE":  False,  # Walk pace: higher = better
    "V06CSTIME1":  True,  # Chair stand time: higher = worse
    "V06400MTIM":  True,   # 400m time: higher = worse
    "V06400MTR":   False,  # 400m metres walked: higher = better

    # Depression
    "V06CESD":     True,   # CES-D: higher = worse

    # Self-reported function and symptoms
    "V06WOMADLR":  True,   # WOMAC disability: higher = worse
    "V06WOMADLL":  True,   # WOMAC disability: higher = worse
    "V06KOOSYMR":  False,  # KOOS symptoms: higher = better
    "V06KOOSYML":  False,  # KOOS symptoms: higher = better

    # Participation (LLDI) - higher = better (per analysis choice)
    "V06LLDIFST":  False,
    "V06LLDILST":  False,

    # Quality of Life
    "V06KOOSQOL": False,    # KOOS Quality of Life: higher = better
}

# Every modelled outcome must carry a direction-of-change convention.
# This mirrors the end-of-1.1 derivation check and catches an outcome
# added to a group but never assigned a direction here, which would
# otherwise KeyError inside the longitudinal evaluation.
_grouped_outcomes = {
    outcome for outcomes in OUTCOME_GROUPS.values() for outcome in outcomes
}
_missing_direction = _grouped_outcomes - set(OUTCOME_HIGHER_IS_WORSE)
assert not _missing_direction, (
    f"OUTCOME_GROUPS outcomes missing from OUTCOME_HIGHER_IS_WORSE: "
    f"{sorted(_missing_direction)}"
)

# Per-outcome stability band: absolute observed change at or below which
# a participant is treated as unchanged (excluded from direction scoring).
#
# Explicit floats are literature values on the CONFIRMED OAI scale (see
# per-line notes). None triggers a runtime fallback in
# resolve_stability_band:
#   - None + an OUTCOME_TEST_RETEST_RELIABILITY entry -> a reliability-
#     based MDC90 (a measurement-error floor); otherwise
#   - None alone -> a distribution-based proxy (0.5 x baseline SD,
#     Norman 2003), a proxy for meaningful change, NOT an error floor.


OUTCOME_STABILITY_BAND: dict[str, float | None] = {
    # KOOS subscales, 0-100 confirmed (higher = better).
    # MDC95, test-retest in OA awaiting arthroplasty. Naylor et al. 2014,
    # BMC Musculoskeletal Disorders, doi:10.1186/1471-2474-15-235.
    "V06KOOSKPR":  20.2,   # KOOS pain
    "V06KOOSKPL":  20.2,
    "V06KOOSYMR":  24.1,   # KOOS symptoms
    "V06KOOSYML":  24.1,
    "V06KOOSQOL":  26.6,   # KOOS Quality of Life

    # ICOAP pain, 0-100 (higher = worse).
    # MCID. Singh et al. 2014, J Rheumatol, doi:10.3899/jrheum.130609.
    # MDC90 (~46.6) rejected: ~half the scale, driven by modest knee ICC.
    "V06ICPTSKR":  18.5,
    "V06ICPTSKL":  18.5,
    # Function
    # 20m walk speed, m/s confirmed (higher = better). Important difference
    # at group/cross-sectional level, not a within-person MDC: source
    # reports 4.1-6.9 m/min, /60 -> 0.07-0.12 m/s, midpoint ~0.10. Gilbert
    # et al. 2020, Arthritis Care Res, doi:10.1002/acr.24159. Noise floor
    # ~0.05 (Motyl et al. 2013, doi:10.1186/1471-2474-14-166).
    "V0620MPACE":  0.1,
    # Repeated chair stand, time for 5 stands, seconds (higher = worse).
    # MCID/MID 1.9 (band 1.7-2.1). COPD population, not OA. Jones et al.
    # 2013, Thorax, doi:10.1136/thoraxjnl-2013-203576. Note: V06CSTIME1 is
    # defined only for participants completing five stands (cs5 = 1);
    # non-completers are missing/floored, biasing the chair-stand direction
    # analysis toward healthier participants.
    "V06CSTIME1":  1.9,    # repeated chair stand time

    # WOMAC disability, 0-68 Likert confirmed (higher = worse).
    # MCID 10.1 (SCB 16.4). Kim et al. 2021, Am J Sports Med,
    # doi:10.1177/03635465211016853.
    "V06WOMADLR":  10.1,
    "V06WOMADLL":  10.1,

    # Depression, CES-D-20, 0-60 (higher = worse).
    # ~10-pt distribution-based MDC95. Primary source is a spinal cord
    # injury (not OA) population: Miller et al. 2007, Spinal Cord,
    # doi:10.1038/sj.sc.3102127. Corroborated by a CES-D-15 anchor MCID
    # of ~11: Haase et al. 2021, J Eval Clin Pract, doi:10.1111/jep.13629.
    "V06CESD":     10.0,

    # Participation, LLFDI Disability component (higher = better; instrument
    # direction confirmed, Jette et al. 2002). Reliability-path MDC90
    # computed at runtime from the classified-sample V06 SD and the Jette
    # et al. 2002 ICCs, doi:10.1093/gerona/57.4.M209. No anchor-based
    # Disability-component MCID exists.
    "V06LLDIFST":  None,   # ICC 0.68 -> see OUTCOME_TEST_RETEST_RELIABILITY
    "V06LLDILST":  None,   # ICC 0.82 -> see OUTCOME_TEST_RETEST_RELIABILITY

    # Baseline-only, excluded from longitudinal
    "V06400MTIM":  None,
    "V06400MTR":   None,
}

# Literature test-retest reliability for outcomes whose stability band is a
# reliability-based MDC90 rather than a fixed literature value. Jette et al.
# 2002, LLFDI Disability component, community-dwelling older adults,
# doi:10.1093/gerona/57.4.M209.
OUTCOME_TEST_RETEST_RELIABILITY: dict[str, float] = {
    "V06LLDILST": 0.82,   # Limitation dimension total
    "V06LLDIFST": 0.68,   # Frequency dimension total
}

# Every outcome carrying a direction must also have a band entry (float
# or explicit None), so a newly added outcome fails loudly here rather
# than KeyError-ing inside the longitudinal loop.
_missing_band = set(OUTCOME_HIGHER_IS_WORSE) - set(OUTCOME_STABILITY_BAND)
assert not _missing_band, (
    f"Outcomes missing from OUTCOME_STABILITY_BAND: {sorted(_missing_band)}"
)

#### Resolve predictors to Visit 08

In [ ]:
def resolve_predictor_to_visit_08(
    column_name: str,
    paired_dataframe: pd.DataFrame,
) -> str:
    """
    Return the Visit 08 column name for a predictor or covariate.

    Two naming regimes coexist in the paired dataframe, because the
    Visit 06 / Visit 08 merge only appends ``_v06`` / ``_v08`` suffixes
    to columns whose names collide:

    - Covariates carry an explicit visit prefix (``V06AGE``, ``V06BMI``,
      ``V06COMORB``). Their Visit 08 counterparts (``V08AGE`` and so on)
      are distinct names, so they do not collide on merge and carry no
      suffix; these are resolved by prefix substitution.
    - Activity predictors and ``sex_female`` share the same name in both
      visit frames, so the merge produced a ``_v08`` suffixed copy;
      these are resolved to that suffixed column.

    Resolving every predictor to its Visit 08 value is required for the
    concurrent transportability evaluation: a model trained on the
    Visit 06 cross-section is applied to an all-Visit-08 feature vector.
    The function raises rather than falling back to the bare column
    name, so a missing Visit 08 column fails loudly instead of silently
    reintroducing a Visit 06 value under a Visit 08 label.

    :param column_name: Predictor or covariate name as used at Visit 06,
        before merge suffixes are applied.
    :param paired_dataframe: Merged Visit 06 / Visit 08 dataframe.
    :returns: The corresponding Visit 08 column name in the paired
        dataframe.
    :raises KeyError: If no Visit 08 column can be resolved.
    """
    if column_name.startswith("V06"):
        visit_08_name = column_name.replace("V06", "V08")
        if visit_08_name in paired_dataframe.columns:
            return visit_08_name
        raise KeyError(
            f"No Visit 08 counterpart '{visit_08_name}' for prefixed "
            f"covariate '{column_name}' in the paired dataframe."
        )

    suffixed_name = f"{column_name}_v08"
    if suffixed_name in paired_dataframe.columns:
        return suffixed_name
    raise KeyError(
        f"Cannot resolve predictor '{column_name}' to a Visit 08 column; "
        f"expected '{suffixed_name}' in the paired dataframe."
    )


def resolve_stability_band(
    *,
    outcome_column: str,
    configured_band: float | None,
    observed_baseline_values: np.ndarray,
    distribution_based_fraction: float = 0.5,
    test_retest_reliability: float | None = None,
) -> float:
    """
    Return the stability band for an outcome, falling back to a
    distribution-based proxy when no literature value is configured.

    Resolution order:

    - If ``configured_band`` is not None, it is returned unchanged. These
      are literature values on the confirmed OAI scale.
    - Else, if ``test_retest_reliability`` is supplied, a minimal
      detectable change at 90% confidence is computed from the baseline
      standard deviation:
      ``MDC90 = 1.645 * sqrt(2) * SD * sqrt(1 - reliability)``. This is a
      genuine measurement-error floor and is preferred when a reliability
      coefficient is available for the instrument.
    - Else, a distribution-based proxy of
      ``distribution_based_fraction * baseline SD`` is returned (the
      Norman "half standard deviation" rule). This approximates a
      minimal important change; it is NOT a measurement-error floor, and
      it is data-dependent (derived from the analysis sample itself).

    :param outcome_column: Name of the outcome, used only for the note.
    :param configured_band: Literature band from OUTCOME_STABILITY_BAND,
        or None to trigger a distribution-based fallback.
    :param observed_baseline_values: Observed Visit 06 values for the
        outcome, used to compute the baseline standard deviation.
    :param distribution_based_fraction: Fraction of the baseline standard
        deviation used for the half-SD proxy. Defaults to 0.5.
    :param test_retest_reliability: Optional literature test-retest
        reliability coefficient; when given, an MDC90 is computed instead
        of the half-SD proxy.
    :returns: The resolved stability band as a float.
    """
    if configured_band is not None:
        return configured_band

    baseline_standard_deviation = float(np.std(observed_baseline_values, ddof=1))

    if test_retest_reliability is not None:
        minimal_detectable_change_90 = (
            1.645
            * np.sqrt(2.0)
            * baseline_standard_deviation
            * np.sqrt(1.0 - test_retest_reliability)
        )
        print(
            f"  NOTE — no configured band for {outcome_column}; using "
            f"MDC90 from baseline SD and reliability "
            f"{test_retest_reliability:.2f} = "
            f"{minimal_detectable_change_90:.3f}."
        )
        return float(minimal_detectable_change_90)

    fallback_band = distribution_based_fraction * baseline_standard_deviation
    print(
        f"  NOTE — no literature band for {outcome_column}; using a "
        f"distribution-based proxy of {distribution_based_fraction:g} x "
        f"baseline SD = {fallback_band:.3f}. Distribution-based MCID "
        f"proxy (Norman half-SD), not a measurement-error floor; "
        f"data-dependent."
    )
    return float(fallback_band)

In [ ]:
def classify_trajectory(
    *,
    delta: np.ndarray,
    higher_is_worse: bool,
    stability_band: float = 0.0,
) -> np.ndarray:
    """
    Classify a delta array into improve, stable, or worsen categories.

    A change within ``stability_band`` of zero (absolute value) is
    labelled ``"stable"``. With the default band of zero this reduces to
    exact-zero behaviour. For outcomes where higher is worse, a negative
    delta beyond the band indicates improvement; where higher is better,
    a positive delta beyond the band indicates improvement.

    :param delta: Array of change values (Visit 08 minus Visit 06).
    :param higher_is_worse: If True, negative delta = improvement.
    :param stability_band: Absolute change at or below which the
        participant is labelled ``"stable"``.
    :returns: String array of ``"improve"``, ``"stable"``, ``"worsen"``.
    """
    is_stable = np.abs(delta) <= stability_band
    trajectory = np.where(is_stable, "stable", "")

    if higher_is_worse:
        trajectory = np.where(~is_stable & (delta < 0), "improve", trajectory)
        trajectory = np.where(~is_stable & (delta > 0), "worsen", trajectory)
    else:
        trajectory = np.where(~is_stable & (delta > 0), "improve", trajectory)
        trajectory = np.where(~is_stable & (delta < 0), "worsen", trajectory)

    return trajectory


def evaluate_direction_agreement(
    *,
    predicted_delta: np.ndarray,
    actual_delta: np.ndarray,
    stability_band: float = 0.0,
) -> tuple[float, int]:
    """
    Proportion of participants whose predicted change moved in the same
    direction as their observed change, among those who genuinely moved.

    A participant is scored only when the absolute OBSERVED change
    exceeds ``stability_band`` (the band is applied to observed change
    only: among participants who moved beyond the band, did the model
    predict the direction correctly?). The metric is valence-free: it
    compares the signs of the two deltas directly and does not depend on
    whether a higher outcome value is better or worse. If no participant
    clears the band, agreement is NaN and the count is zero.

    :param predicted_delta: Predicted Visit 08 minus observed Visit 06.
    :param actual_delta: Observed Visit 08 minus observed Visit 06.
    :param stability_band: Absolute observed change below which a
        participant is treated as unchanged and excluded from scoring.
    :returns: Tuple of (direction agreement, number scored).
    """
    scored = np.abs(actual_delta) > stability_band
    number_scored = int(scored.sum())

    if number_scored == 0:
        return float("nan"), 0

    agreement = float(
        np.mean(np.sign(predicted_delta[scored]) == np.sign(actual_delta[scored]))
    )
    return agreement, number_scored


def majority_class_direction_baseline(
    *,
    actual_delta: np.ndarray,
    stability_band: float = 0.0,
) -> tuple[float, int, int]:
    """
    Compute the no-information direction accuracy for a single outcome.

    Among participants who moved beyond the stability band, this is the
    accuracy achieved by always predicting the modal observed direction
    of change. It is the direction-of-change analogue of the
    no-information rate in classification and is the reference against
    which the model's direction-of-change accuracy must be read: only a
    model that clears this baseline carries genuine information about
    the direction of change, rather than exploiting an imbalance between
    improvers and worseners in the cohort.

    The baseline is valence-free. It is computed from the sign of the
    observed change alone and does not depend on whether a higher value
    is clinically better or worse. Participants whose absolute observed
    change does not exceed ``stability_band`` are excluded, matching the
    denominator used for the model's direction-of-change accuracy.

    :param actual_delta: Observed Visit 08 minus observed Visit 06.
    :param stability_band: Absolute observed change at or below which a
        participant is treated as unchanged and excluded from scoring.
    :returns: Tuple of (baseline accuracy, modal observed direction as
        +1 or -1, number of participants scored). If no participant
        clears the band, the accuracy is NaN, the modal direction is 0
        and the count is zero.
    """
    has_moved = np.abs(actual_delta) > stability_band
    number_scored = int(has_moved.sum())

    if number_scored == 0:
        return float("nan"), 0, 0

    observed_signs = np.sign(actual_delta[has_moved])
    number_increased = int(np.sum(observed_signs > 0))
    number_decreased = int(np.sum(observed_signs < 0))

    modal_count = max(number_increased, number_decreased)
    baseline_accuracy = modal_count / number_scored
    modal_observed_direction = 1 if number_increased >= number_decreased else -1

    return float(baseline_accuracy), modal_observed_direction, number_scored


def evaluate_longitudinal_predictions(
    *,
    outcome_column: str,
    refitted_model_bundle: dict,
    paired_dataframe: pd.DataFrame,
    outcome_stability_band: dict[str, float | None],
    outcome_test_retest_reliability: dict[str, float],
) -> dict:
    """
    Generate Visit 08 predictions for a single outcome and evaluate
    direction-of-change accuracy against observed Visit 08 values.

    The Visit 08 predictor matrix is assembled using the covariate and
    activity predictor columns stored in the refitted model bundle. For
    linear models the Visit 06 scaler is applied to activity predictors.
    For tree-based models no scaling is applied. An empty covariate block
    (activity-only bundles) is handled cleanly: the covariate matrix is
    zero-width and drops out of the horizontal stack.

    Direction-of-change accuracy is the proportion of participants for
    whom the predicted direction of change (predicted Visit 08 minus
    observed Visit 06) matches the actual direction of change (observed
    Visit 08 minus observed Visit 06), among those whose observed change
    exceeds the per-outcome stability band (resolved via
    resolve_stability_band). It is reported alongside the majority-class
    baseline accuracy, which is the accuracy of always predicting the
    modal observed direction on the same set of movers, and the
    difference between the two. The metric is valence-free: it compares
    the signs of the deltas directly and does not depend on whether a
    higher value is better or worse.

    :param outcome_column: Name of the outcome variable.
    :param refitted_model_bundle: Output of
        refit_winning_model_on_full_visit_06 for this outcome.
    :param paired_dataframe: Merged Visit 06 / Visit 08 dataframe with
        suffixes _v06 and _v08 on overlapping columns.
    :param outcome_stability_band: Per-outcome stability band table; a
        literature float on the confirmed scale, or None to trigger a
        distribution-based fallback in resolve_stability_band.
    :param outcome_test_retest_reliability: Literature test-retest
        reliability per outcome; when an entry exists for an outcome
        that has no configured band, resolve_stability_band returns a
        reliability-based MDC90 instead of the half-SD proxy.
    :returns: Dictionary with keys outcome_column, n_paired,
        n_direction_evaluated, direction_of_change_accuracy,
        direction_accuracy_ci_low, direction_accuracy_ci_high,
        majority_class_baseline_accuracy, modal_observed_direction,
        direction_accuracy_above_baseline, stability_band,
        test_r2_visit_08, rmse_visit_08, mae_visit_08.
    """
    fitted_model = refitted_model_bundle["fitted_model"]
    scaler = refitted_model_bundle["scaler"]
    covariate_columns = refitted_model_bundle["covariate_columns"]
    activity_predictor_columns = refitted_model_bundle["activity_predictor_columns"]

    resolved_covariate_columns = [
        resolve_predictor_to_visit_08(c, paired_dataframe)
        for c in covariate_columns
    ]
    resolved_activity_columns = [
        resolve_predictor_to_visit_08(c, paired_dataframe)
        for c in activity_predictor_columns
    ]

    outcome_v06_column = (
        f"{outcome_column}_v06"
        if f"{outcome_column}_v06" in paired_dataframe.columns
        else outcome_column
    )
    outcome_v08_column = (
        f"{outcome_column}_v08"
        if f"{outcome_column}_v08" in paired_dataframe.columns
        else outcome_column.replace("V06", "V08")
    )

    all_required_columns = (
        resolved_covariate_columns
        + resolved_activity_columns
        + [outcome_v06_column, outcome_v08_column]
    )

    complete_cases = paired_dataframe[all_required_columns].dropna()
    number_of_complete_cases = len(complete_cases)

    print(f"  Complete paired cases : {number_of_complete_cases:,}")

    covariate_matrix = complete_cases[resolved_covariate_columns].values
    activity_matrix = complete_cases[resolved_activity_columns].values

    if scaler is not None:
        activity_matrix_scaled = scaler.transform(activity_matrix)
        full_matrix = np.hstack([covariate_matrix, activity_matrix_scaled])
    else:
        full_matrix = np.hstack([covariate_matrix, activity_matrix])

    visit_08_predictions = fitted_model.predict(full_matrix)
    observed_visit_06 = complete_cases[outcome_v06_column].values
    observed_visit_08 = complete_cases[outcome_v08_column].values

    test_r2_visit_08 = r2_score(observed_visit_08, visit_08_predictions)
    rmse_visit_08 = float(np.sqrt(mean_squared_error(observed_visit_08, visit_08_predictions)))
    mae_visit_08 = float(mean_absolute_error(observed_visit_08, visit_08_predictions))

    actual_delta = observed_visit_08 - observed_visit_06
    predicted_delta = visit_08_predictions - observed_visit_06

    stability_band = resolve_stability_band(
        outcome_column=outcome_column,
        configured_band=outcome_stability_band.get(outcome_column),
        observed_baseline_values=observed_visit_06,
        test_retest_reliability=outcome_test_retest_reliability.get(outcome_column),
    )

    direction_of_change_accuracy, number_direction_evaluated = evaluate_direction_agreement(
        predicted_delta=predicted_delta,
        actual_delta=actual_delta,
        stability_band=stability_band,
    )

    (
        majority_class_baseline_accuracy,
        modal_observed_direction,
        _,
    ) = majority_class_direction_baseline(
        actual_delta=actual_delta,
        stability_band=stability_band,
    )
    direction_accuracy_above_baseline = (
        direction_of_change_accuracy - majority_class_baseline_accuracy
    )

    if number_direction_evaluated > 0:
        number_of_correct_directions = round(
            direction_of_change_accuracy * number_direction_evaluated
        )
        direction_accuracy_ci_low, direction_accuracy_ci_high = (
            compute_direction_accuracy_interval(
                number_of_correct_directions=number_of_correct_directions,
                number_of_evaluated_participants=number_direction_evaluated,
            )
        )
    else:
        direction_accuracy_ci_low = float("nan")
        direction_accuracy_ci_high = float("nan")

    print(
        f"  Visit 08 R\u00b2                    : {test_r2_visit_08:.4f}\n"
        f"  Visit 08 RMSE                  : {rmse_visit_08:.4f}\n"
        f"  Visit 08 MAE                   : {mae_visit_08:.4f}\n"
        f"  Stability band                 : {stability_band:.4f}\n"
        f"  Direction-of-change accuracy   : {direction_of_change_accuracy:.4f} "
        f"(n={number_direction_evaluated})\n"
        f"  Direction 95% CI (Wilson)      : "
        f"[{direction_accuracy_ci_low:.4f}, {direction_accuracy_ci_high:.4f}]\n"
        f"  Majority-class baseline        : {majority_class_baseline_accuracy:.4f}\n"
        f"  Accuracy above baseline        : {direction_accuracy_above_baseline:+.4f}"
    )

    return {
        "outcome_column": outcome_column,
        "n_paired": number_of_complete_cases,
        "n_direction_evaluated": number_direction_evaluated,
        "direction_of_change_accuracy": direction_of_change_accuracy,
        "direction_accuracy_ci_low": direction_accuracy_ci_low,
        "direction_accuracy_ci_high": direction_accuracy_ci_high,
        "majority_class_baseline_accuracy": majority_class_baseline_accuracy,
        "modal_observed_direction": modal_observed_direction,
        "direction_accuracy_above_baseline": direction_accuracy_above_baseline,
        "stability_band": stability_band,
        "test_r2_visit_08": test_r2_visit_08,
        "rmse_visit_08": rmse_visit_08,
        "mae_visit_08": mae_visit_08,
    }


def compute_direction_accuracy_interval(
    *,
    number_of_correct_directions: int,
    number_of_evaluated_participants: int,
    confidence_level: float = 0.95,
) -> tuple[float, float]:
    """Return the Wilson score interval for direction-of-change accuracy.

    The Wilson interval is used instead of the normal approximation because
    the evaluated subsets are small and the observed proportions lie close to
    one, where the normal approximation produces upper bounds above unity.

    :param number_of_correct_directions: Movers whose predicted and observed
        change had the same sign.
    :param number_of_evaluated_participants: Total movers evaluated.
    :param confidence_level: Nominal coverage of the interval.
    :returns: Lower and upper bound of the interval.
    """
    test_result = binomtest(
        k=number_of_correct_directions, n=number_of_evaluated_participants
    )
    interval = test_result.proportion_ci(
        confidence_level=confidence_level, method="wilson"
    )
    return interval.low, interval.high

In [ ]:
longitudinal_results: list[dict] = []

for outcome_column, refitted_model_bundle in refitted_models.items():
    print(f"\n{'=' * 60}")
    print(f"Outcome: {outcome_column}  [{refitted_model_bundle['best_model']}]")
    print(f"{'=' * 60}")

    try:
        result = evaluate_longitudinal_predictions(
            outcome_column=outcome_column,
            refitted_model_bundle=refitted_model_bundle,
            paired_dataframe=paired_dataframe,
            outcome_stability_band=OUTCOME_STABILITY_BAND,
            outcome_test_retest_reliability=OUTCOME_TEST_RETEST_RELIABILITY,
        )
        longitudinal_results.append(result)

    except Exception as error:
        print(f"  Skipped: {error}")

# Activity-only tier: identical evaluation on the covariate-free bundles.
longitudinal_results_activity_only: list[dict] = []

for outcome_column, refitted_model_bundle in refitted_models_activity_only.items():
    print(f"\n{'=' * 60}")
    print(f"Outcome (activity-only): {outcome_column}  [{refitted_model_bundle['best_model']}]")
    print(f"{'=' * 60}")

    try:
        result = evaluate_longitudinal_predictions(
            outcome_column=outcome_column,
            refitted_model_bundle=refitted_model_bundle,
            paired_dataframe=paired_dataframe,
            outcome_stability_band=OUTCOME_STABILITY_BAND,
            outcome_test_retest_reliability=OUTCOME_TEST_RETEST_RELIABILITY,
        )
        longitudinal_results_activity_only.append(result)

    except Exception as error:
        print(f"  Skipped: {error}")

In [ ]:
# Combined-model summary
longitudinal_summary_table = (
    pd.DataFrame(longitudinal_results)
    .sort_values("direction_of_change_accuracy", ascending=False)
    .reset_index(drop=True)
)

longitudinal_summary_table.to_csv(
    prediction_output / "stage_8_longitudinal_validation.csv",
    index=False,
)

# Like-for-like contrast: does activity-only direction-of-change hold up
# next to the combined model? Join the two tiers on outcome so the
# comparison lives in a single table. The majority-class baseline is a
# property of the observed change alone, so it is identical across tiers
# and is carried once from the combined side; the margin over baseline
# differs by tier and is kept for both.
_combined = pd.DataFrame(longitudinal_results)[
    [
        "outcome_column",
        "n_paired",
        "direction_of_change_accuracy",
        "direction_accuracy_ci_low",
        "direction_accuracy_ci_high",
        "majority_class_baseline_accuracy",
        "direction_accuracy_above_baseline",
        "test_r2_visit_08",
    ]
]
_activity_only = pd.DataFrame(longitudinal_results_activity_only)[
    [
        "outcome_column",
        "direction_of_change_accuracy",
        "direction_accuracy_ci_low",
        "direction_accuracy_ci_high",
        "direction_accuracy_above_baseline",
        "test_r2_visit_08",
    ]
]
longitudinal_comparison_table = (
    _combined.merge(
        _activity_only,
        on="outcome_column",
        suffixes=("_combined", "_activity_only"),
    )
    .sort_values("direction_of_change_accuracy_combined", ascending=False)
    .reset_index(drop=True)
)

longitudinal_comparison_table.to_csv(
    prediction_output / "stage_8_longitudinal_combined_vs_activity_only.csv",
    index=False,
)

_display_table = longitudinal_comparison_table.copy()
_display_table["doc_95pct_ci_combined"] = [
    f"[{low:.2f}, {high:.2f}]"
    for low, high in zip(
        _display_table["direction_accuracy_ci_low_combined"],
        _display_table["direction_accuracy_ci_high_combined"],
    )
]

print("\nStage 8 — Longitudinal validation: combined vs activity-only")
print("(direction-of-change accuracy with 95% CI vs majority-class baseline "
      "and Visit 08 R\u00b2, sorted by combined DoC):")
print(_display_table[
    [
        "outcome_column",
        "n_paired",
        "majority_class_baseline_accuracy",
        "direction_of_change_accuracy_combined",
        "doc_95pct_ci_combined",
        "direction_accuracy_above_baseline_combined",
        "direction_of_change_accuracy_activity_only",
        "direction_accuracy_above_baseline_activity_only",
        "test_r2_visit_08_combined",
        "test_r2_visit_08_activity_only",
    ]
].to_string(index=False))

## 2.7 Per-participant prediction export

In [ ]:
def build_per_participant_prediction_table(
    *,
    outcome_column: str,
    refitted_model_bundle: dict,
    paired_dataframe: pd.DataFrame,
    outcome_higher_is_worse: dict[str, bool],
    outcome_stability_band: dict[str, float | None],
    outcome_test_retest_reliability: dict[str, float],
) -> pd.DataFrame:
    """
    Build a per-participant prediction table for a single outcome.

    For each participant with complete data the table contains:
    - participant ID
    - observed Visit 06 value
    - predicted Visit 08 value
    - observed Visit 08 value
    - actual delta (observed V08 minus observed V06)
    - predicted delta (predicted V08 minus observed V06)
    - actual trajectory (improve / stable / worsen)
    - predicted trajectory (improve / stable / worsen)
    - scored: True if the participant moved beyond the stability band
    - correct: sign agreement (1.0/0.0) for scored movers, NaN otherwise;
      its NaN-skipping mean equals the headline direction accuracy

    :param outcome_column: Name of the outcome variable.
    :param refitted_model_bundle: Output of
        refit_winning_model_on_full_visit_06 for this outcome.
    :param paired_dataframe: Merged Visit 06 / Visit 08 dataframe with
        suffixes _v06 and _v08 on overlapping columns.
    :param outcome_higher_is_worse: Mapping of outcome column name to
        whether a higher value indicates worse status.
    :param outcome_stability_band: Per-outcome stability band table
        (literature float or None for a distribution-based fallback).
    :param outcome_test_retest_reliability: Literature test-retest
        reliability per outcome; enables the MDC90 band path in
        resolve_stability_band for outcomes with no configured band.
    :returns: DataFrame with one row per participant.
    """
    higher_is_worse = outcome_higher_is_worse[outcome_column]

    fitted_model = refitted_model_bundle["fitted_model"]
    scaler = refitted_model_bundle["scaler"]
    covariate_columns = refitted_model_bundle["covariate_columns"]
    activity_predictor_columns = refitted_model_bundle["activity_predictor_columns"]

    resolved_covariate_columns = [
        resolve_predictor_to_visit_08(c, paired_dataframe)
        for c in covariate_columns
    ]
    resolved_activity_columns = [
        resolve_predictor_to_visit_08(c, paired_dataframe)
        for c in activity_predictor_columns
    ]

    outcome_v06_column = (
        f"{outcome_column}_v06"
        if f"{outcome_column}_v06" in paired_dataframe.columns
        else outcome_column
    )
    outcome_v08_column = (
        f"{outcome_column}_v08"
        if f"{outcome_column}_v08" in paired_dataframe.columns
        else outcome_column.replace("V06", "V08")
    )

    all_required_columns = (
        ["ID"]
        + resolved_covariate_columns
        + resolved_activity_columns
        + [outcome_v06_column, outcome_v08_column]
    )

    complete_cases = paired_dataframe[all_required_columns].dropna()

    covariate_matrix = complete_cases[resolved_covariate_columns].values
    activity_matrix = complete_cases[resolved_activity_columns].values

    if scaler is not None:
        activity_matrix_scaled = scaler.transform(activity_matrix)
        full_matrix = np.hstack([covariate_matrix, activity_matrix_scaled])
    else:
        full_matrix = np.hstack([covariate_matrix, activity_matrix])

    visit_08_predictions = fitted_model.predict(full_matrix)
    observed_visit_06 = complete_cases[outcome_v06_column].values
    observed_visit_08 = complete_cases[outcome_v08_column].values

    actual_delta = observed_visit_08 - observed_visit_06
    predicted_delta = visit_08_predictions - observed_visit_06

    stability_band = resolve_stability_band(
        outcome_column=outcome_column,
        configured_band=outcome_stability_band.get(outcome_column),
        observed_baseline_values=observed_visit_06,
        test_retest_reliability=outcome_test_retest_reliability.get(outcome_column),
    )

    actual_trajectory = classify_trajectory(
        delta=actual_delta,
        higher_is_worse=higher_is_worse,
        stability_band=stability_band,
    )
    predicted_trajectory = classify_trajectory(
        delta=predicted_delta,
        higher_is_worse=higher_is_worse,
        stability_band=stability_band,
    )

    # Canonical direction agreement, defined identically to the headline
    # metric (evaluate_direction_agreement): scored only for movers whose
    # observed change clears the stability band, as raw sign agreement.
    # Non-movers are excluded (NaN), NOT assigned a banded "stable" -- the
    # predicted delta is not magnitude-calibrated, so it must never be
    # thresholded against a clinical band. The NaN-skipping mean of this
    # column equals the outcome's direction-of-change accuracy.
    is_mover = np.abs(actual_delta) > stability_band
    direction_correct = np.where(
        np.sign(predicted_delta) == np.sign(actual_delta), 1.0, 0.0
    )
    direction_correct[~is_mover] = np.nan

    prediction_table = pd.DataFrame({
        "ID": complete_cases["ID"].values,
        "observed_v06": observed_visit_06,
        "predicted_v08": visit_08_predictions,
        "observed_v08": observed_visit_08,
        "actual_delta": actual_delta,
        "predicted_delta": predicted_delta,
        "actual_trajectory": actual_trajectory,
        "predicted_trajectory": predicted_trajectory,
        "scored": is_mover,
        "correct": direction_correct,
    })

    return prediction_table

In [ ]:
per_participant_tables: dict[str, pd.DataFrame] = {}

for outcome_column, refitted_model_bundle in refitted_models.items():
    prediction_table = build_per_participant_prediction_table(
        outcome_column=outcome_column,
        refitted_model_bundle=refitted_model_bundle,
        paired_dataframe=paired_dataframe,
        outcome_higher_is_worse=OUTCOME_HIGHER_IS_WORSE,
        outcome_stability_band=OUTCOME_STABILITY_BAND,
        outcome_test_retest_reliability=OUTCOME_TEST_RETEST_RELIABILITY,
    )

    per_participant_tables[outcome_column] = prediction_table

    output_filename = f"predictions_{outcome_column}.csv"
    prediction_table.to_csv(
        prediction_output / output_filename,
        index=False,
    )

print(
    f"Per-participant prediction tables built and saved for "
    f"{len(per_participant_tables)} outcomes."
)
print(
    f"\nTo inspect a specific outcome:\n"
    f"  per_participant_tables['V06KOOSKPR'].head(20)"
)

In [ ]:
all_outcomes_per_participant = pd.concat(
    [
        prediction_table.assign(outcome=outcome_column)
        for outcome_column, prediction_table in per_participant_tables.items()
    ],
    ignore_index=True,
)

all_outcomes_per_participant = all_outcomes_per_participant[
    [
        "ID",
        "outcome",
        "observed_v06",
        "predicted_v08",
        "observed_v08",
        "actual_delta",
        "predicted_delta",
        "actual_trajectory",
        "predicted_trajectory",
        "correct",
    ]
]

all_outcomes_per_participant.to_csv(
    prediction_output / "predictions_all_outcomes.csv",
    index=False,
)

print(
    f"Saved predictions for {all_outcomes_per_participant['ID'].nunique():,} "
    f"participants across {all_outcomes_per_participant['outcome'].nunique()} "
    f"outcomes — {len(all_outcomes_per_participant):,} rows total."
)

In [ ]:
# EXPLORATORY descriptive view only -- NOT an accuracy metric. This
# cross-tabs the banded improve/stable/worsen trajectories, which
# threshold the PREDICTED delta against the clinical band. The predicted
# delta is not magnitude-calibrated (the model was trained on the V06
# cross-section, not to reproduce change magnitude), so this view can
# mislabel systematic under/over-prediction as "stable". The headline
# direction-of-change accuracy (movers-only sign agreement) is the
# canonical figure; use this only to eyeball case distributions.
trajectory_accuracy_summary = (
    all_outcomes_per_participant
    .groupby(["outcome", "actual_trajectory", "predicted_trajectory"])
    .size()
    .reset_index(name="count")
    .sort_values(["outcome", "actual_trajectory", "predicted_trajectory"])
)

print("Trajectory cross-tab (EXPLORATORY -- assumes a calibrated predicted delta):")
print(trajectory_accuracy_summary.to_string(index=False))

## 2.8 Refit KL grade classifiers on full Visit 06

Refits both the Random Forest and XGBoost classifiers from Stage 6 on
the complete Visit 06 dataset (no train/test split) in preparation for
longitudinal KL grade prediction at Visit 08.

In [ ]:
def refit_kl_grade_classifier_on_full_visit_06(
    *,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    structural_covariate_column: str,
    model_name: str,
    random_state: int = 42,
) -> dict:
    """
    Refit a KL grade classifier on the full Visit 06 dataset.

    No train/test split is applied. The held-out test set kappa from
    Stage 6 already provides the unbiased performance estimate.
    Refitting on the full dataset maximises the training signal before
    generalising to Visit 08.

    :param dataframe: Full Visit 06 modelling dataframe.
    :param activity_predictor_columns: Activity predictor column names.
    :param demographic_covariate_columns: Demographic covariate column
        names. Must not contain KL grade.
    :param structural_covariate_column: KL grade column name. Used as
        the outcome — not included as a predictor.
    :param model_name: Either ``"random_forest"`` or ``"xgboost"``.
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys ``model_name``, ``fitted_classifier``,
        ``all_predictor_columns``, and ``n_fitted``.
    :raises ValueError: If ``model_name`` is not recognised.
    """
    if model_name not in {"random_forest", "xgboost"}:
        raise ValueError(
            f"Unrecognised model_name '{model_name}'. "
            f"Expected 'random_forest' or 'xgboost'."
        )

    # Training vintage is Visit 06 by design: the classifier learns the
    # activity-to-KL-grade mapping on the Visit 06 cross-section and is
    # then applied to an all-Visit-08 feature vector at evaluation time
    # (see resolve_predictor_to_visit_08). These covariates stay on V06.
    all_predictor_columns = (
        list(demographic_covariate_columns) + activity_predictor_columns
    )
    all_required_columns = all_predictor_columns + [structural_covariate_column]

    complete_cases = dataframe[all_required_columns].dropna()
    number_of_complete_cases = len(complete_cases)

    print(
        f"  Refitting on full Visit 06 — {number_of_complete_cases:,} complete cases"
    )

    predictor_matrix = complete_cases[all_predictor_columns].values
    outcome_classes = complete_cases[structural_covariate_column].astype(int).values

    if model_name == "random_forest":
        fitted_classifier = RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=10,
            class_weight="balanced",
            n_jobs=-1,
            random_state=random_state,
        )
        fitted_classifier.fit(predictor_matrix, outcome_classes)

    else:
        fitted_classifier = InverseFrequencyWeightedXGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softmax",
            num_class=int(np.max(outcome_classes)) + 1,
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        )
        fitted_classifier.fit(predictor_matrix, outcome_classes)

    print(f"  {model_name.replace('_', ' ').title()} refitted successfully")

    return {
        "model_name": model_name,
        "fitted_classifier": fitted_classifier,
        "all_predictor_columns": all_predictor_columns,
        "n_fitted": number_of_complete_cases,
    }

#### Execute KL grade classifier refit

In [ ]:
refitted_kl_grade_classifiers: dict[str, dict] = {}

for model_name in ["random_forest", "xgboost"]:
    print(f"\n{'=' * 60}")
    print(f"Model: {model_name}")
    print(f"{'=' * 60}")

    refitted_kl_grade_classifiers[model_name] = refit_kl_grade_classifier_on_full_visit_06(
        dataframe=summary_metrics_06,
        activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
        demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
        structural_covariate_column=STRUCTURAL_COVARIATE_COLUMN,
        model_name=model_name,
        random_state=RANDOM_STATE,
    )

print(f"\nKL grade classifiers refitted and ready for Visit 08 prediction.")

## 2.9 Longitudinal KL grade prediction and evaluation

For each refitted classifier, Visit 08 KL grade is predicted from the
Visit 08 predictor matrix.

In [ ]:
def evaluate_longitudinal_kl_grade_predictions(
    *,
    model_name: str,
    refitted_classifier_bundle: dict,
    paired_dataframe: pd.DataFrame,
    structural_covariate_column: str,
) -> dict:
    """
    Generate Visit 08 KL grade predictions and evaluate against observed
    Visit 08 KL grade values.

    Direction-of-change accuracy for KL grade is computed as the
    proportion of participants for whom the predicted direction of grade
    change (predicted V08 minus observed V06) matches the actual
    direction (observed V08 minus observed V06), among participants whose
    grade changed at Visit 08. It is reported alongside the
    majority-class baseline accuracy (the accuracy of always predicting
    the modal observed direction on the same set of movers) and the
    difference between the two, computed with the same shared helpers as
    the continuous outcomes.

    :param model_name: Name of the classifier (``"random_forest"`` or
        ``"xgboost"``).
    :param refitted_classifier_bundle: Output of
        refit_kl_grade_classifier_on_full_visit_06.
    :param paired_dataframe: Merged Visit 06 / Visit 08 dataframe with
        suffixes _v06 and _v08 on overlapping columns.
    :param structural_covariate_column: KL grade column name as it
        appears in Visit 06 (before merge suffixes are applied).
    :returns: Dictionary with keys model_name, n_paired,
        n_direction_evaluated, direction_of_change_accuracy,
        direction_accuracy_ci_low, direction_accuracy_ci_high,
        majority_class_baseline_accuracy, modal_observed_direction,
        direction_accuracy_above_baseline, test_kappa, test_accuracy,
        test_mae_grade, confusion_matrix.
    """
    fitted_classifier = refitted_classifier_bundle["fitted_classifier"]
    all_predictor_columns = refitted_classifier_bundle["all_predictor_columns"]

    resolved_predictor_columns = [
        resolve_predictor_to_visit_08(column, paired_dataframe)
        for column in all_predictor_columns
    ]

    kl_grade_v06_column = (
        f"{structural_covariate_column}_v06"
        if f"{structural_covariate_column}_v06" in paired_dataframe.columns
        else structural_covariate_column
    )
    kl_grade_v08_column = (
        f"{structural_covariate_column}_v08"
        if f"{structural_covariate_column}_v08" in paired_dataframe.columns
        else structural_covariate_column
    )

    all_required_columns = (
        resolved_predictor_columns
        + [kl_grade_v06_column, kl_grade_v08_column]
    )

    complete_cases = paired_dataframe[all_required_columns].dropna()
    number_of_complete_cases = len(complete_cases)

    print(f"  Complete paired cases : {number_of_complete_cases:,}")

    predictor_matrix = complete_cases[resolved_predictor_columns].values
    observed_kl_v06 = complete_cases[kl_grade_v06_column].astype(int).values
    observed_kl_v08 = complete_cases[kl_grade_v08_column].astype(int).values

    visit_08_predictions = fitted_classifier.predict(predictor_matrix).astype(int)

    test_kappa = cohen_kappa_score(
        y1=observed_kl_v08,
        y2=visit_08_predictions,
        weights="quadratic",
    )
    test_accuracy = accuracy_score(observed_kl_v08, visit_08_predictions)
    test_mae_grade = float(np.mean(np.abs(observed_kl_v08 - visit_08_predictions)))

    actual_delta = observed_kl_v08 - observed_kl_v06
    predicted_delta = visit_08_predictions - observed_kl_v06

    # Reuse the shared helpers: identical direction-of-change definition
    # as the continuous outcomes, with the zero-mover guard and the
    # majority-class baseline included. The default stability band of
    # zero treats any non-zero grade change as a move.
    direction_of_change_accuracy, number_direction_evaluated = (
        evaluate_direction_agreement(
            predicted_delta=predicted_delta,
            actual_delta=actual_delta,
        )
    )
    (
        majority_class_baseline_accuracy,
        modal_observed_direction,
        _,
    ) = majority_class_direction_baseline(actual_delta=actual_delta)
    direction_accuracy_above_baseline = (
        direction_of_change_accuracy - majority_class_baseline_accuracy
    )

    if number_direction_evaluated > 0:
        number_of_correct_directions = round(
            direction_of_change_accuracy * number_direction_evaluated
        )
        direction_accuracy_ci_low, direction_accuracy_ci_high = (
            compute_direction_accuracy_interval(
                number_of_correct_directions=number_of_correct_directions,
                number_of_evaluated_participants=number_direction_evaluated,
            )
        )
    else:
        direction_accuracy_ci_low = float("nan")
        direction_accuracy_ci_high = float("nan")

    confusion_matrix = pd.crosstab(
        pd.Series(observed_kl_v08, name="actual"),
        pd.Series(visit_08_predictions, name="predicted"),
    )

    print(
        f"  Test kappa (QWK)               : {test_kappa:.4f}\n"
        f"  Test accuracy                  : {test_accuracy:.4f}\n"
        f"  Test MAE (grade)               : {test_mae_grade:.4f}\n"
        f"  Direction-of-change accuracy   : {direction_of_change_accuracy:.4f} "
        f"(n={number_direction_evaluated})\n"
        f"  Direction 95% CI (Wilson)      : "
        f"[{direction_accuracy_ci_low:.4f}, {direction_accuracy_ci_high:.4f}]\n"
        f"  Majority-class baseline        : {majority_class_baseline_accuracy:.4f}\n"
        f"  Accuracy above baseline        : {direction_accuracy_above_baseline:+.4f}\n"
        f"\n  Confusion matrix (rows: actual, columns: predicted):\n"
        f"{confusion_matrix}"
    )

    return {
        "model_name": model_name,
        "n_paired": number_of_complete_cases,
        "n_direction_evaluated": number_direction_evaluated,
        "direction_of_change_accuracy": direction_of_change_accuracy,
        "direction_accuracy_ci_low": direction_accuracy_ci_low,
        "direction_accuracy_ci_high": direction_accuracy_ci_high,
        "majority_class_baseline_accuracy": majority_class_baseline_accuracy,
        "modal_observed_direction": modal_observed_direction,
        "direction_accuracy_above_baseline": direction_accuracy_above_baseline,
        "test_kappa": test_kappa,
        "test_accuracy": test_accuracy,
        "test_mae_grade": test_mae_grade,
        "confusion_matrix": confusion_matrix,
    }

#### Execute longitudinal KL grade evaluation


In [ ]:
longitudinal_kl_grade_results: dict[str, dict] = {}

for model_name, refitted_classifier_bundle in refitted_kl_grade_classifiers.items():
    print(f"\n{'=' * 60}")
    print(f"Model: {model_name}")
    print(f"{'=' * 60}")

    longitudinal_kl_grade_results[model_name] = evaluate_longitudinal_kl_grade_predictions(
        model_name=model_name,
        refitted_classifier_bundle=refitted_classifier_bundle,
        paired_dataframe=paired_dataframe,
        structural_covariate_column=STRUCTURAL_COVARIATE_COLUMN,
    )

#### Longitudinal KL grade summary table


In [ ]:
print(f"\n{'=' * 70}")
print("Stage 8 (KL grade) — Longitudinal prediction summary (Visit 06 → Visit 08)")
print(f"{'=' * 70}")

print(
    f"\n{'Model':<20} {'Test kappa':>12} {'Accuracy':>12} "
    f"{'MAE grade':>12} {'Dir. accuracy':>15} {'DoC 95% CI':>18} {'Baseline':>12} "
    f"{'Above base':>12} {'n dir.':>8}"
)
print("-" * 129)

for model_name, result in longitudinal_kl_grade_results.items():
    ci_text = (
        f"[{result['direction_accuracy_ci_low']:.2f}, "
        f"{result['direction_accuracy_ci_high']:.2f}]"
    )
    print(
        f"{model_name.replace('_', ' ').title():<20} "
        f"{result['test_kappa']:>12.4f} "
        f"{result['test_accuracy']:>12.4f} "
        f"{result['test_mae_grade']:>12.4f} "
        f"{result['direction_of_change_accuracy']:>15.4f} "
        f"{ci_text:>18} "
        f"{result['majority_class_baseline_accuracy']:>12.4f} "
        f"{result['direction_accuracy_above_baseline']:>+12.4f} "
        f"{result['n_direction_evaluated']:>8}"
    )

longitudinal_kl_grade_summary_table = pd.DataFrame(
    [
        {
            "model": model_name,
            "n_paired": result["n_paired"],
            "test_kappa": result["test_kappa"],
            "test_accuracy": result["test_accuracy"],
            "test_mae_grade": result["test_mae_grade"],
            "direction_of_change_accuracy": result["direction_of_change_accuracy"],
            "direction_accuracy_ci_low": result["direction_accuracy_ci_low"],
            "direction_accuracy_ci_high": result["direction_accuracy_ci_high"],
            "majority_class_baseline_accuracy": result["majority_class_baseline_accuracy"],
            "direction_accuracy_above_baseline": result["direction_accuracy_above_baseline"],
            "modal_observed_direction": result["modal_observed_direction"],
            "n_direction_evaluated": result["n_direction_evaluated"],
        }
        for model_name, result in longitudinal_kl_grade_results.items()
    ]
)
longitudinal_kl_grade_summary_table.to_csv(
    prediction_output / "stage_8_kl_grade_longitudinal.csv",
    index=False,
)

## 2.10 Confirmatory inference: incremental R² permutation test

In [ ]:
def assemble_visit_06_training_arrays(
    *,
    outcome_column: str,
    combined_model_bundle: dict,
    visit_06_dataframe: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Assemble the Visit 06 training arrays in the feature space the model
    consumes.

    The covariate block is always passed through unscaled, mirroring the
    covariate passthrough in the combined refit. The activity block is
    standardised with the bundle's Visit 06 scaler for linear models and
    left raw for tree-based models. Permuting the rows of a
    standardised activity block is equivalent to permuting then
    standardising, because a row permutation leaves each column's mean
    and standard deviation unchanged, so the stored scaler remains valid
    under permutation.

    :param outcome_column: Name of the outcome variable.
    :param combined_model_bundle: Combined-model bundle from
        refit_winning_model_on_full_visit_06.
    :param visit_06_dataframe: Full Visit 06 modelling dataframe.
    :returns: Tuple of covariate block, activity block (model space),
        and outcome vector, all restricted to complete cases.
    """
    scaler = combined_model_bundle["scaler"]
    covariate_columns = combined_model_bundle["covariate_columns"]
    activity_predictor_columns = combined_model_bundle["activity_predictor_columns"]

    required_columns = (
        covariate_columns + activity_predictor_columns + [outcome_column]
    )
    complete_cases = visit_06_dataframe[required_columns].dropna()

    covariate_block = complete_cases[covariate_columns].values
    raw_activity_block = complete_cases[activity_predictor_columns].values
    outcome_vector = complete_cases[outcome_column].values

    activity_block = (
        scaler.transform(raw_activity_block)
        if scaler is not None
        else raw_activity_block
    )

    return covariate_block, activity_block, outcome_vector


def assemble_visit_08_evaluation_arrays(
    *,
    outcome_column: str,
    combined_model_bundle: dict,
    paired_dataframe: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Assemble the Visit 08 evaluation arrays on a single shared set of
    complete cases.

    Both the covariate-only and the combined model are evaluated on the
    same rows so that their R2 values are directly comparable and their
    difference is a clean incremental quantity. The complete-case mask
    matches the one used in evaluate_longitudinal_predictions (covariates,
    activity, and both visit outcomes present), so the combined R2 here
    reproduces the Visit 08 R2 already reported in Section 2.6.

    :param outcome_column: Name of the outcome variable.
    :param combined_model_bundle: Combined-model bundle from
        refit_winning_model_on_full_visit_06.
    :param paired_dataframe: Merged Visit 06 / Visit 08 dataframe.
    :returns: Tuple of covariate block, activity block (model space),
        and observed Visit 08 outcome vector, all on the shared
        complete-case set.
    """
    scaler = combined_model_bundle["scaler"]
    covariate_columns = combined_model_bundle["covariate_columns"]
    activity_predictor_columns = combined_model_bundle["activity_predictor_columns"]

    resolved_covariate_columns = [
        resolve_predictor_to_visit_08(column_name, paired_dataframe)
        for column_name in covariate_columns
    ]
    resolved_activity_columns = [
        resolve_predictor_to_visit_08(column_name, paired_dataframe)
        for column_name in activity_predictor_columns
    ]

    outcome_v06_column = (
        f"{outcome_column}_v06"
        if f"{outcome_column}_v06" in paired_dataframe.columns
        else outcome_column
    )
    outcome_v08_column = (
        f"{outcome_column}_v08"
        if f"{outcome_column}_v08" in paired_dataframe.columns
        else outcome_column.replace("V06", "V08")
    )

    required_columns = (
        resolved_covariate_columns
        + resolved_activity_columns
        + [outcome_v06_column, outcome_v08_column]
    )
    complete_cases = paired_dataframe[required_columns].dropna()

    covariate_block = complete_cases[resolved_covariate_columns].values
    raw_activity_block = complete_cases[resolved_activity_columns].values
    observed_outcome = complete_cases[outcome_v08_column].values

    activity_block = (
        scaler.transform(raw_activity_block)
        if scaler is not None
        else raw_activity_block
    )

    return covariate_block, activity_block, observed_outcome


def run_incremental_r2_permutation_test(
    *,
    outcome_column: str,
    combined_model_bundle: dict,
    visit_06_dataframe: pd.DataFrame,
    paired_dataframe: pd.DataFrame,
    permutation_count: int,
    random_state: int,
) -> dict:
    """
    Test whether the activity block adds held-out Visit 08 variance
    beyond a same-family covariate-only baseline, via an activity-block
    permutation null.

    The observed statistic is the incremental R2 on Visit 08, defined as
    the combined-model R2 minus the covariate-only R2, both trained on
    the full Visit 06 set and evaluated on the real Visit 08 set. The
    covariate-only baseline reuses the combined model's family and
    hyperparameters (obtained by cloning the fitted combined estimator),
    fitted on the covariates alone.

    The null distribution is built by permuting the rows of the Visit 06
    activity block against the aligned covariates and outcome, refitting
    the combined model on each permuted training set, and re-evaluating
    on the untouched Visit 08 set. The covariate-only R2 is invariant
    under this permutation and is therefore computed once. Refitting on
    permuted training data (rather than permuting at evaluation time)
    calibrates the null against training-time overfitting, which a
    held-out increment can otherwise conceal.

    The one-sided permutation p-value uses the add-one estimator
    ``(1 + count) / (1 + permutation_count)``, one-sided because only a
    positive increment is of interest.

    :param outcome_column: Name of the outcome variable.
    :param combined_model_bundle: Combined-model bundle from
        refit_winning_model_on_full_visit_06.
    :param visit_06_dataframe: Full Visit 06 modelling dataframe.
    :param paired_dataframe: Merged Visit 06 / Visit 08 dataframe.
    :param permutation_count: Number of permutations (the B in the
        p-value estimator).
    :param random_state: Seed for the permutation index generator.
    :returns: Dictionary with the outcome name, winning family, the two
        observed R2 values, the observed incremental R2, the permutation
        count, the permutation p-value, and the training and evaluation
        sample sizes.
    """
    covariate_train, activity_train, outcome_train = assemble_visit_06_training_arrays(
        outcome_column=outcome_column,
        combined_model_bundle=combined_model_bundle,
        visit_06_dataframe=visit_06_dataframe,
    )
    covariate_eval, activity_eval, observed_outcome = assemble_visit_08_evaluation_arrays(
        outcome_column=outcome_column,
        combined_model_bundle=combined_model_bundle,
        paired_dataframe=paired_dataframe,
    )

    template_estimator = combined_model_bundle["fitted_model"]

    # Observed combined model.
    combined_estimator = clone(template_estimator)
    combined_estimator.fit(
        np.hstack([covariate_train, activity_train]),
        outcome_train,
    )
    r2_combined_observed = r2_score(
        observed_outcome,
        combined_estimator.predict(np.hstack([covariate_eval, activity_eval])),
    )

    # Same-family covariate-only baseline (invariant under permutation).
    covariate_only_estimator = clone(template_estimator)
    covariate_only_estimator.fit(covariate_train, outcome_train)
    r2_covariate_only = r2_score(
        observed_outcome,
        covariate_only_estimator.predict(covariate_eval),
    )

    observed_incremental_r2 = r2_combined_observed - r2_covariate_only

    # Permutation null: shuffle the training activity block only.
    permutation_generator = np.random.default_rng(random_state)
    number_of_training_rows = activity_train.shape[0]
    count_at_least_as_extreme = 0

    for _ in range(permutation_count):
        shuffled_row_index = permutation_generator.permutation(number_of_training_rows)
        permuted_activity_train = activity_train[shuffled_row_index]

        null_estimator = clone(template_estimator)
        null_estimator.fit(
            np.hstack([covariate_train, permuted_activity_train]),
            outcome_train,
        )
        r2_combined_null = r2_score(
            observed_outcome,
            null_estimator.predict(np.hstack([covariate_eval, activity_eval])),
        )
        incremental_r2_null = r2_combined_null - r2_covariate_only

        if incremental_r2_null >= observed_incremental_r2:
            count_at_least_as_extreme += 1

    permutation_p_value = (1 + count_at_least_as_extreme) / (1 + permutation_count)

    print(
        f"  {outcome_column:<40} "
        f"ΔR²={observed_incremental_r2:+.4f}  p={permutation_p_value:.4f}"
    )

    return {
        "outcome_column": outcome_column,
        "best_model": combined_model_bundle["best_model"],
        "r2_covariate_only": r2_covariate_only,
        "r2_combined_observed": r2_combined_observed,
        "observed_incremental_r2": observed_incremental_r2,
        "permutation_count": permutation_count,
        "permutation_p_value": permutation_p_value,
        "n_train": number_of_training_rows,
        "n_eval": int(observed_outcome.shape[0]),
    }


def benjamini_hochberg_adjust(*, p_values: np.ndarray) -> np.ndarray:
    """
    Compute Benjamini-Hochberg adjusted p-values, preserving input order.

    :param p_values: One-dimensional array of raw p-values.
    :returns: Array of BH-adjusted p-values in the original order,
        clipped to the unit interval.
    """
    raw_p_values = np.asarray(p_values, dtype=float)
    number_of_tests = raw_p_values.size

    ascending_order = np.argsort(raw_p_values)
    ranked_p_values = raw_p_values[ascending_order]

    rank_positions = np.arange(1, number_of_tests + 1)
    scaled_p_values = ranked_p_values * number_of_tests / rank_positions

    # Enforce monotonicity from the largest rank downwards.
    monotone_p_values = np.minimum.accumulate(scaled_p_values[::-1])[::-1]
    monotone_p_values = np.clip(monotone_p_values, 0.0, 1.0)

    adjusted_p_values = np.empty_like(monotone_p_values)
    adjusted_p_values[ascending_order] = monotone_p_values
    return adjusted_p_values

In [ ]:
PRIMARY_OUTCOME_FAMILY: dict[str, str] = {
    # KOOS Pain subscale (bilateral)
    "V06KOOSKPR":  "koos_pain",
    "V06KOOSKPL":  "koos_pain",

    # KOOS Symptoms subscale (bilateral; distinct from pain and function)
    "V06KOOSYMR":  "koos_symptoms",
    "V06KOOSYML":  "koos_symptoms",

    # ICOAP (bilateral)
    "V06ICPTSKR":  "icoap",
    "V06ICPTSKL":  "icoap",

    # 20m walk performance test (pace)
    "V0620MPACE":  "walk_20m",

    # Repeated chair stand performance test
    "V06CSTIME1":   "chair_stand",

    # CES-D depression scale
    "V06CESD":     "cesd",

    # WOMAC Physical Function / Disability (bilateral)
    "V06WOMADLR":  "womac_pf",
    "V06WOMADLL":  "womac_pf",

    # KOOS Quality of Life
    "V06KOOSQOL":  "koos_qol",

    # LLDI: Limitation component (instrumental total)
    "V06LLDIFST":  "lldi_frequency",
    "V06LLDILST":  "lldi_limitation",
}

FALSE_DISCOVERY_RATE = 0.05

In [ ]:
# Guard: every outcome that can reach Stage 9 must carry a family, and
# the map should not list outcomes that are never modelled. The tested
# set is the longitudinal outcomes (grouped outcomes minus the two
# baseline-only 400m measures, which are not reassessed at Visit 08).
_baseline_only_outcomes = {"V06400MTIM", "V06400MTR"}
_testable_outcomes = {
    outcome
    for outcomes in OUTCOME_GROUPS.values()
    for outcome in outcomes
} - _baseline_only_outcomes

_unmapped_outcomes = _testable_outcomes - set(PRIMARY_OUTCOME_FAMILY)
_dead_map_entries = set(PRIMARY_OUTCOME_FAMILY) - _testable_outcomes

assert not _unmapped_outcomes, (
    f"Outcomes reachable by Stage 9 but missing from "
    f"PRIMARY_OUTCOME_FAMILY (they would fall through to 'exploratory' "
    f"and be dropped from FDR correction): {sorted(_unmapped_outcomes)}"
)
assert not _dead_map_entries, (
    f"PRIMARY_OUTCOME_FAMILY lists outcomes that are never modelled: "
    f"{sorted(_dead_map_entries)}"
)

In [ ]:
STAGE_9_PERMUTATION_COUNT = 1000  

print(f"\n{'=' * 70}")
print("Stage 9 — incremental R² permutation test (Visit 06 → Visit 08)")
print(f"{'=' * 70}\n")

stage_9_rows: list[dict] = []
for outcome_column, combined_model_bundle in refitted_models.items():
    stage_9_rows.append(
        run_incremental_r2_permutation_test(
            outcome_column=outcome_column,
            combined_model_bundle=combined_model_bundle,
            visit_06_dataframe=summary_metrics_06,
            paired_dataframe=paired_dataframe,
            permutation_count=STAGE_9_PERMUTATION_COUNT,
            random_state=RANDOM_STATE,
        )
    )

stage_9_table = pd.DataFrame(stage_9_rows)
stage_9_table["family"] = (
    stage_9_table["outcome_column"].map(PRIMARY_OUTCOME_FAMILY).fillna("exploratory")
)
stage_9_table["bh_adjusted_p_value"] = np.nan
stage_9_table["reject_null"] = False

for family_label, family_rows in stage_9_table.groupby("family"):
    if family_label == "exploratory":
        continue
    adjusted = benjamini_hochberg_adjust(
        p_values=family_rows["permutation_p_value"].values
    )
    stage_9_table.loc[family_rows.index, "bh_adjusted_p_value"] = adjusted
    stage_9_table.loc[family_rows.index, "reject_null"] = adjusted <= FALSE_DISCOVERY_RATE

stage_9_table = stage_9_table.sort_values(
    ["family", "observed_incremental_r2"], ascending=[True, False]
).reset_index(drop=True)

stage_9_table.to_csv(prediction_output / "stage_9_confirmatory_inference.csv", index=False)

print(f"\n{'-' * 70}")
print(stage_9_table[
    [
        "outcome_column", "family", "best_model",
        "r2_covariate_only", "r2_combined_observed", "observed_incremental_r2",
        "permutation_p_value", "bh_adjusted_p_value", "reject_null",
    ]
].to_string(index=False))

# 3. Visualisation of predictive model performance




In [ ]:
"""Build the three publication figures for the accelerometry manuscript.

Each figure is written to its own pair of files, a vector version for
submission and a raster version for preview.

Figure 1 reports how often each candidate accelerometry predictor was retained
across the exploratory outcome models.

Figure 2 reports temporal transportability. For every outcome it shows the
held-out estimate of the algorithm that performed best at the 48 month visit and
the estimate obtained when that same algorithm is applied at the 72 month visit.

Figure 3 reports the confirmatory comparison at the 72 month visit between a
covariates-only model and a model that adds the activity block, together with the
incremental coefficient of determination and its permutation test result.
"""

from pathlib import Path

import matplotlib
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch

PALETTE_LIGHTEST = "#C9E3DA"
PALETTE_LIGHT = "#A9D3C6"
PALETTE_MIDDLE = "#3F8E76"
PALETTE_DARK = "#3B8574"
PALETTE_DARKEST = "#0F4340"

COLOUR_HIGHLIGHT = PALETTE_DARK
COLOUR_HIGHLIGHT_LIGHT = PALETTE_LIGHT
COLOUR_NEUTRAL = "#8A8A8A"
COLOUR_NEUTRAL_LIGHT = "#C6C6C6"
COLOUR_INK = "#2F2F2F"
COLOUR_RULE = "#DCDCDC"

ALGORITHM_COLUMNS = (
    "ridge_test_r2",
    "elastic_net_test_r2",
    "random_forest_test_r2",
    "xgboost_test_r2",
)

ALGORITHM_DISPLAY_NAMES = {
    "ridge": "Ridge",
    "elastic_net": "Elastic net",
    "random_forest": "Random forest",
    "xgboost": "XGBoost",
}

OUTCOME_DISPLAY_NAMES = {
    "V0620MPACE": "20 m walk pace",
    "V06400MTIM": "400 m walk time",
    "V06400MTR": "400 m walk trial",
    "V06CSTIME1": "Chair-stand time",
    "V06CESD": "CES-D depression",
    "V06LLDIFST": "LLFDI frequency",
    "V06LLDILST": "LLFDI limitation",
    "V06KOOSQOL": "KOOS quality of life",
    "V06KOOSKPR": "KOOS pain (right)",
    "V06KOOSKPL": "KOOS pain (left)",
    "V06KOOSYMR": "KOOS symptoms (right)",
    "V06KOOSYML": "KOOS symptoms (left)",
    "V06WOMADLR": "WOMAC function (right)",
    "V06WOMADLL": "WOMAC function (left)",
    "V06ICPTSKR": "ICOAP (right)",
    "V06ICPTSKL": "ICOAP (left)",
}

FEATURE_DISPLAY_NAMES = {
    "interdaily_stability": "Interdaily stability",
    "iv_weekday": "Intradaily variability (weekday)",
    "iv_weekend": "Intradaily variability (weekend)",
    "acrophase": "Acrophase (all days)",
    "acrophase_mean_curve_weekday": "Acrophase (weekday mean curve)",
    "acrophase_mean_curve_weekend": "Acrophase (weekend mean curve)",
    "mesor_mean_curve_weekday": "Mesor (weekday mean curve)",
    "mesor_mean_curve_weekend": "Mesor (weekend mean curve)",
    "activity_onset_minute_mean": "Activity onset time, mean",
    "activity_onset_minute_sd": "Activity onset time, within-person variability",
    "activity_offset_minute_mean": "Activity offset time, mean",
    "activity_offset_minute_sd": "Activity offset time, within-person variability",
    "mean_sedentary_bout_total_minutes": "Sedentary bouts, total minutes",
    "mean_sedentary_bout_max_duration": "Sedentary bouts, longest duration",
    "mean_sedentary_bout_mean_duration": "Sedentary bouts, mean duration",
    "mean_sedentary_bout_count": "Sedentary bouts, count",
    "mean_mvpa_bout_count": "Moderate to vigorous bouts, count",
    "mean_mvpa_bout_mean_duration": "Moderate to vigorous bouts, mean duration",
    "mean_light_bout_total_minutes": "Light activity bouts, total minutes",
    "mean_light_bout_max_duration": "Light activity bouts, longest duration",
    "mean_light_bout_mean_duration": "Light activity bouts, mean duration",
    "V06AACNT": "Mean daily counts",
    "V06AALTMNT": "Light activity counts",
    "V06AAMDMNT": "Moderate activity counts",
    "V06AAMVMNT": "Moderate to vigorous counts",
    "V06AAVMNT": "Vigorous activity counts",
    "wear_duration_mean": "Monitor wear duration, mean",
}

RETAINED_COLUMN_RIGHT_EDGE = 1.235
COMBINED_INCREMENT_COLUMN_RIGHT_EDGE = 0.410
ACTIVITY_INCREMENT_COLUMN_RIGHT_EDGE = 0.505

FEATURE_DISPLAY_NAMES = {
    "interdaily_stability": "Interdaily stability",
    "iv_weekday": "Intradaily variability (weekday)",
    "iv_weekend": "Intradaily variability (weekend)",
    "acrophase": "Acrophase (all days)",
    "acrophase_mean_curve_weekday": "Acrophase (weekday)",
    "acrophase_mean_curve_weekend": "Acrophase (weekend)",
    "mesor_mean_curve_weekday": "Mesor (weekday)",
    "mesor_mean_curve_weekend": "Mesor (weekend)",
    "activity_onset_minute_mean": "Activity onset time, mean",
    "activity_onset_minute_sd": "Activity onset time, SD",
    "activity_offset_minute_mean": "Activity offset time, mean",
    "activity_offset_minute_sd": "Activity offset time, SD",
    "mean_sedentary_bout_total_minutes": "Sedentary bouts, total minutes",
    "mean_sedentary_bout_max_duration": "Sedentary bouts, longest duration",
    "mean_sedentary_bout_mean_duration": "Sedentary bouts, mean duration",
    "mean_sedentary_bout_count": "Sedentary bout count",
    "mean_mvpa_bout_count": "Moderate to vigorous bouts count",
    "mean_mvpa_bout_mean_duration": "Moderate to vigorous bouts, mean duration",
    "mean_light_bout_total_minutes": "Light activity bouts, total minutes",
    "mean_light_bout_max_duration": "Light activity bouts, longest duration",
    "mean_light_bout_mean_duration": "Light activity bouts, mean duration",
    "V06AACNT": "Mean daily counts",
    "V06AALTMNT": "Light activity counts",
    "V06AAMDMNT": "Moderate activity counts",
    "V06AAMVMNT": "Moderate to vigorous counts",
    "V06AAVMNT": "Vigorous activity counts",
    "wear_duration_mean": "Monitor wear duration, mean",
}

INCREMENT_COLUMN_RIGHT_EDGE = 0.335
PREDICTOR_COUNT_COLUMN_RIGHT_EDGE = 0.425
RETAINED_COLUMN_RIGHT_EDGE = 1.235
COMBINED_INCREMENT_COLUMN_RIGHT_EDGE = 0.410

FAMILY_HARMONIC = "Harmonic and rhythmic"
FAMILY_TIMING = "Onset and offset"
FAMILY_INTENSITY = "Intensity and bouts"

FEATURE_FAMILY_NAMES = {
    "interdaily_stability": FAMILY_HARMONIC,
    "iv_weekday": FAMILY_HARMONIC,
    "iv_weekend": FAMILY_HARMONIC,
    "acrophase": FAMILY_HARMONIC,
    "acrophase_mean_curve_weekday": FAMILY_HARMONIC,
    "acrophase_mean_curve_weekend": FAMILY_HARMONIC,
    "mesor_mean_curve_weekday": FAMILY_HARMONIC,
    "mesor_mean_curve_weekend": FAMILY_HARMONIC,
    "activity_onset_minute_mean": FAMILY_TIMING,
    "activity_onset_minute_sd": FAMILY_TIMING,
    "activity_offset_minute_mean": FAMILY_TIMING,
    "activity_offset_minute_sd": FAMILY_TIMING,
    "mean_sedentary_bout_total_minutes": FAMILY_INTENSITY,
    "mean_sedentary_bout_max_duration": FAMILY_INTENSITY,
    "mean_sedentary_bout_mean_duration": FAMILY_INTENSITY,
    "mean_sedentary_bout_count": FAMILY_INTENSITY,
    "mean_mvpa_bout_count": FAMILY_INTENSITY,
    "mean_mvpa_bout_mean_duration": FAMILY_INTENSITY,
    "mean_light_bout_total_minutes": FAMILY_INTENSITY,
    "mean_light_bout_max_duration": FAMILY_INTENSITY,
    "mean_light_bout_mean_duration": FAMILY_INTENSITY,
    "V06AACNT": FAMILY_INTENSITY,
    "V06AALTMNT": FAMILY_INTENSITY,
    "V06AAMDMNT": FAMILY_INTENSITY,
    "V06AAMVMNT": FAMILY_INTENSITY,
    "V06AAVMNT": FAMILY_INTENSITY,
    "wear_duration_mean": FAMILY_INTENSITY,
}

FEATURE_FAMILY_COLOURS = {
    FAMILY_HARMONIC: PALETTE_DARKEST,
    FAMILY_TIMING: PALETTE_MIDDLE,
    FAMILY_INTENSITY: PALETTE_LIGHTEST,
}

def draw_rounded_horizontal_bars(
    *,
    axes,
    values,
    row_positions,
    colours,
    bar_height: float = 0.66,
    rounding_fraction: float = 0.4,
) -> None:
    """Draw borderless horizontal bars with visually circular rounded ends.

    The rounding radius is specified in inches on the rendered page and then
    converted into data units, so the corners stay circular regardless of how
    different the two axis scales are. Bars shorter than twice the radius have
    the radius reduced so that a short bar is not distorted into a lozenge.

    :param axes: The axes to draw on. Its limits must already be final.
    :param values: Bar lengths in data units.
    :param row_positions: Vertical centre of each bar in data units.
    :param colours: Fill colour for each bar.
    :param bar_height: Bar thickness in data units.
    :param rounding_fraction: Corner radius as a fraction of the bar thickness.
    :return: Nothing. Patches are added to the axes.
    """
    figure = axes.get_figure()
    figure.canvas.draw()
    axes_extent = axes.get_window_extent()
    width_in_inches = axes_extent.width / figure.dpi
    height_in_inches = axes_extent.height / figure.dpi
    x_lower, x_upper = axes.get_xlim()
    y_lower, y_upper = axes.get_ylim()
    x_units_per_inch = (x_upper - x_lower) / width_in_inches
    y_units_per_inch = (y_upper - y_lower) / height_in_inches
    mutation_aspect = y_units_per_inch / x_units_per_inch
    nominal_radius = rounding_fraction * bar_height / mutation_aspect

    for value, row_position, colour in zip(values, row_positions, colours):
        if value <= 0:
            continue
        radius = min(nominal_radius, value / 2)
        axes.add_patch(
            FancyBboxPatch(
                (0.0, row_position - bar_height / 2),
                value,
                bar_height,
                boxstyle=f"round,pad=0,rounding_size={radius}",
                mutation_aspect=mutation_aspect,
                facecolor=colour,
                edgecolor="none",
                linewidth=0.0,
                zorder=3,
            )
        )


def apply_house_style() -> None:
    """Set the shared Matplotlib style for both figures.

    :return: Nothing. The global Matplotlib parameters are modified in place.
    """

    plt.style.use("default")

    plt.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Liberation Sans", "DejaVu Sans"],
            "font.size": 8.5,
            "axes.edgecolor": COLOUR_RULE,
            "axes.labelcolor": COLOUR_INK,
            "text.color": COLOUR_INK,
            "xtick.color": COLOUR_NEUTRAL,
            "ytick.color": COLOUR_INK,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8.5,
            "axes.linewidth": 0.6,
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "savefig.edgecolor": "white",
            "legend.facecolor": "white",
            "savefig.facecolor": "white",
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def draw_predictor_retention_figure(
    *,
    selection_frequency_path: Path,
    output_directory: Path,
    file_stem: str,
) -> list[Path]:
    """Draw the predictor retention figure and write it to disk.

    :param selection_frequency_path: Path to the selection frequency export.
    :param output_directory: Directory that will receive the rendered files.
    :param file_stem: Filename without extension.
    :return: Paths of the written files, vector formats first (PDF,
        EPS, then PNG).
    """
    selection_frequency = (
        pd.read_csv(selection_frequency_path)
        .sort_values("selection_proportion", ascending=True)
        .reset_index(drop=True)
    )
    row_count = len(selection_frequency)
    figure, axes = plt.subplots(figsize=(7.8, 0.25 * row_count + 2.0))

    families = [
        FEATURE_FAMILY_NAMES[name] for name in selection_frequency["feature"]
    ]
    for row_position, record in selection_frequency.iterrows():
        axes.text(
            RETAINED_COLUMN_RIGHT_EDGE,
            row_position,
            f"{int(record['selection_count'])}/{int(record['outcomes_modelled'])}",
            fontsize=7.2,
            va="center",
            ha="right",
            color=COLOUR_NEUTRAL,
        )

    axes.set_yticks(range(row_count))
    axes.set_yticklabels(
        [
            FEATURE_DISPLAY_NAMES.get(name, name)
            for name in selection_frequency["feature"]
        ]
    )
    axes.set_ylim(-0.8, row_count + 0.35)
    axes.set_xlim(0.0, 1.27)
    axes.set_xticks([0.0, 0.25, 0.50, 0.75, 1.0])
    axes.set_xlabel("Proportion of outcomes in which the predictor was retained")
    axes.text(
        RETAINED_COLUMN_RIGHT_EDGE,
        row_count + 0.05,
        "Retained",
        fontsize=7.5,
        ha="right",
        color=COLOUR_NEUTRAL,
    )
    axes.legend(
        handles=[
            Patch(facecolor=FEATURE_FAMILY_COLOURS[family], label=family)
            for family in (FAMILY_HARMONIC, FAMILY_TIMING, FAMILY_INTENSITY)
        ],
        loc="lower left",
        bbox_to_anchor=(0.0, 1.005),
        frameon=False,
        fontsize=7.8,
        ncol=2,
        handlelength=1.0,
        handletextpad=0.4,
    )

    axes.spines["bottom"].set_bounds(0.0, 1.0)
    for spine_name in ("top", "right", "left"):
        axes.spines[spine_name].set_visible(False)
    axes.tick_params(axis="y", length=0)
    axes.set_axisbelow(True)

    figure.tight_layout(rect=(0, 0.01, 1, 1))
    axes.barh(
        y=range(row_count),
        width=selection_frequency["selection_proportion"],
        height=0.66,
        color=[FEATURE_FAMILY_COLOURS[family] for family in families],
        edgecolor="none",
        linewidth=0.0,
        zorder=3,
    )
    return _save_figure(
        figure=figure, output_directory=output_directory, file_stem=file_stem
    )


def load_transportability_table(
    *,
    model_comparison_path: Path,
    longitudinal_validation_path: Path,
) -> pd.DataFrame:
    """Join the 48 month model comparison to the 72 month validation results.

    Outcomes without a 72 month estimate are dropped, because the figure compares
    the two visits row by row and a missing value has no position.

    :param model_comparison_path: Path to the 48 month model comparison export.
    :param longitudinal_validation_path: Path to the 72 month validation export.
    :return: One row per outcome, ordered by the 72 month estimate.
    """
    model_comparison = pd.read_csv(model_comparison_path)
    longitudinal_validation = pd.read_csv(longitudinal_validation_path)
    combined = model_comparison.merge(
        longitudinal_validation[["outcome_column", "test_r2_visit_08"]],
        left_on="outcome",
        right_on="outcome_column",
        how="inner",
    )
    return combined.sort_values("test_r2_visit_08", ascending=True).reset_index(
        drop=True
    )


def draw_transportability_figure(
    *,
    transportability_table: pd.DataFrame,
    output_directory: Path,
    file_stem: str,
) -> list[Path]:
    """Draw the 48 to 72 month transportability figure and write it to disk.

    :param transportability_table: Output of :func:`load_transportability_table`.
    :param output_directory: Directory that will receive the rendered files.
    :param file_stem: Filename without extension.
    :return: Paths of the written files, vector formats first (PDF,
        EPS, then PNG).
    """
    row_count = len(transportability_table)
    figure, axes = plt.subplots(figsize=(7.2, 0.36 * row_count + 1.7))

    axes.axvline(0.0, color=COLOUR_NEUTRAL, linewidth=0.8, zorder=2)

    for row_position, record in transportability_table.iterrows():
        axes.axhline(row_position, color=COLOUR_RULE, linewidth=0.5, zorder=1)
        axes.plot(
            [record["best_test_r2"], record["test_r2_visit_08"]],
            [row_position, row_position],
            color=PALETTE_LIGHT,
            linewidth=1.4,
            solid_capstyle="round",
            zorder=4,
        )
        axes.plot(
            record["best_test_r2"],
            row_position,
            marker="o",
            markersize=6.2,
            markerfacecolor=PALETTE_DARK,
            markeredgecolor="white",
            markeredgewidth=0.8,
            zorder=5,
        )
        axes.plot(
            record["test_r2_visit_08"],
            row_position,
            marker="D",
            markersize=5.4,
            markerfacecolor=COLOUR_INK,
            markeredgecolor="white",
            markeredgewidth=0.8,
            zorder=6,
        )

    axes.set_yticks(range(row_count))
    axes.set_yticklabels(
        [
            OUTCOME_DISPLAY_NAMES.get(name, name)
            for name in transportability_table["outcome"]
        ]
    )
    axes.set_ylim(-0.8, row_count - 0.2)
    axes.set_xlim(-0.17, 0.32)
    axes.set_xlabel("Held-out coefficient of determination")
    axes.legend(
        handles=[
            Line2D(
                [],
                [],
                marker="o",
                linestyle="none",
                markersize=6.2,
                markerfacecolor=PALETTE_DARK,
                markeredgecolor="white",
                label="48 months, best performing model",
            ),
            Line2D(
                [],
                [],
                marker="D",
                linestyle="none",
                markersize=5.4,
                markerfacecolor=COLOUR_INK,
                markeredgecolor="white",
                label="72 months, same model applied",
            ),
        ],
        loc="lower left",
        bbox_to_anchor=(0.0, 1.005),
        frameon=False,
        fontsize=7.8,
        handletextpad=0.4,
    )

    for spine_name in ("top", "right", "left"):
        axes.spines[spine_name].set_visible(False)
    axes.tick_params(axis="y", length=0)

    figure.tight_layout(rect=(0, 0.01, 1, 1))
    return _save_figure(
        figure=figure, output_directory=output_directory, file_stem=file_stem
    )


def draw_confirmatory_increment_figure(
    *,
    confirmatory_inference_path: Path,
    output_directory: Path,
    file_stem: str,
) -> list[Path]:
    """Draw the covariates-only against combined comparison and write it to disk.

    :param confirmatory_inference_path: Path to the permutation inference export.
    :param output_directory: Directory that will receive the rendered files.
    :param file_stem: Filename without extension.
    :return: Paths of the written files, vector formats first (PDF,
        EPS, then PNG).
    """
    inference = (
        pd.read_csv(confirmatory_inference_path)
        .sort_values("r2_combined_observed", ascending=True)
        .reset_index(drop=True)
    )
    row_count = len(inference)
    figure, axes = plt.subplots(figsize=(8.4, 0.36 * row_count + 1.9))

    axes.axvline(0.0, color=COLOUR_NEUTRAL, linewidth=0.8, zorder=2)

    for row_position, record in inference.iterrows():
        activity_block_retained = bool(record["reject_null"])
        combined_colour = (
            PALETTE_DARKEST if activity_block_retained else COLOUR_NEUTRAL
        )
        axes.axhline(row_position, color=COLOUR_RULE, linewidth=0.5, zorder=1)
        axes.plot(
            [record["r2_covariate_only"], record["r2_combined_observed"]],
            [row_position, row_position],
            color=PALETTE_LIGHT if activity_block_retained else COLOUR_NEUTRAL_LIGHT,
            linewidth=1.4,
            solid_capstyle="round",
            zorder=4,
        )
        axes.plot(
            record["r2_covariate_only"],
            row_position,
            marker="o",
            markersize=5.4,
            markerfacecolor="white",
            markeredgecolor=COLOUR_NEUTRAL,
            markeredgewidth=1.0,
            zorder=5,
        )
        axes.plot(
            record["r2_combined_observed"],
            row_position,
            marker="o",
            markersize=6.2,
            markerfacecolor=combined_colour,
            markeredgecolor="white",
            markeredgewidth=0.8,
            zorder=6,
        )
        significance_mark = "*" if activity_block_retained else ""
        axes.text(
            ACTIVITY_INCREMENT_COLUMN_RIGHT_EDGE,
            row_position,
            f"\u0394{record['observed_incremental_r2']:+.3f}{significance_mark}",
            fontsize=7.5,
            va="center",
            ha="right",
            color=COLOUR_INK if activity_block_retained else COLOUR_NEUTRAL,
        )

    axes.set_yticks(range(row_count))
    axes.set_yticklabels(
        [
            OUTCOME_DISPLAY_NAMES.get(name, name)
            for name in inference["outcome_column"]
        ]
    )
    axes.set_ylim(-0.8, row_count + 0.35)
    axes.set_xlim(-0.35, 0.53)
    axes.set_xticks([-0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3])
    axes.set_xlabel("Held-out coefficient of determination at 72 months")
    axes.text(
        ACTIVITY_INCREMENT_COLUMN_RIGHT_EDGE,
        row_count + 0.05,
        "Increment",
        fontsize=7.5,
        ha="right",
        color=COLOUR_NEUTRAL,
    )
    axes.legend(
        handles=[
            Line2D(
                [],
                [],
                marker="o",
                linestyle="none",
                markersize=5.4,
                markerfacecolor="white",
                markeredgecolor=COLOUR_NEUTRAL,
                label="Covariates only",
            ),
            Line2D(
                [],
                [],
                marker="o",
                linestyle="none",
                markersize=6.2,
                markerfacecolor=PALETTE_DARKEST,
                markeredgecolor="white",
                label="Combined, activity block retained after correction",
            ),
            Line2D(
                [],
                [],
                marker="o",
                linestyle="none",
                markersize=6.2,
                markerfacecolor=COLOUR_NEUTRAL,
                markeredgecolor="white",
                label="Combined, null retained",
            ),
        ],
        loc="lower left",
        bbox_to_anchor=(0.0, 1.005),
        frameon=False,
        fontsize=7.8,
        handletextpad=0.4,
    )

    axes.spines["bottom"].set_bounds(-0.30, 0.30)
    for spine_name in ("top", "right", "left"):
        axes.spines[spine_name].set_visible(False)
    axes.tick_params(axis="y", length=0)

    figure.text(
        0.01,
        0.006,
        "\u0394 = incremental coefficient of determination contributed by the "
        "activity block. * null rejected after Benjamini and Hochberg correction "
        "within outcome family.",
        fontsize=7.2,
        color=COLOUR_NEUTRAL,
    )
    figure.tight_layout(rect=(0, 0.04, 1, 1))
    return _save_figure(
        figure=figure, output_directory=output_directory, file_stem=file_stem
    )


def add_rounded_background(
    *,
    figure,
    colour: str,
    corner_radius_in_inches: float = 0.11,
) -> None:
    """Place a rounded, tinted panel behind everything the figure draws.

    The figure's own background patch is a plain rectangle and cannot be rounded,
    so a rounded patch is laid over the full canvas instead and the canvas itself
    is left white. The corner radius is given in inches and converted using the
    ratio of the figure's two dimensions, which keeps the corners circular rather
    than elliptical on a non-square figure.

    :param figure: The figure to place the panel on.
    :param colour: Fill colour of the panel.
    :param corner_radius_in_inches: Corner radius on the printed page.
    :return: Nothing. The patch is added to the figure.
    """
    width_in_inches, height_in_inches = figure.get_size_inches()
    figure.patch.set_facecolor("white")
    figure.add_artist(
        FancyBboxPatch(
            (0.0, 0.0),
            1.0,
            1.0,
            boxstyle=(
                "round,pad=0,"
                f"rounding_size={corner_radius_in_inches / width_in_inches}"
            ),
            mutation_aspect=width_in_inches / height_in_inches,
            transform=figure.transFigure,
            facecolor=colour,
            edgecolor="none",
            zorder=-10,
        )
    )


def draw_combined_transport_and_increment_figure(
    *,
    model_comparison_path: Path,
    confirmatory_inference_path: Path,
    output_directory: Path,
    file_stem: str,
) -> list[Path]:
    """Draw transportability and the confirmatory increment as one two-panel figure.

    The figure is laid out at the final printed width so that nothing is scaled
    down when it is placed in the manuscript.

    Both panels plot the same quantity, a held-out coefficient of determination.
    The right panel needs extra horizontal room for the increment column, so its
    axis limits and its width share are widened by the same proportion. That keeps
    the number of page units per unit of the coefficient identical in the two
    panels, which is what makes the spans visually comparable.

    :param model_comparison_path: Path to the 48 month model comparison export.
    :param confirmatory_inference_path: Path to the permutation inference export.
    :param output_directory: Directory that will receive the rendered files.
    :param file_stem: Filename without extension.
    :return: Paths of the written files, vector formats first (PDF,
        EPS, then PNG).
    """
    inference = pd.read_csv(confirmatory_inference_path)
    model_comparison = pd.read_csv(model_comparison_path)
    table = (
        inference.merge(
            model_comparison[["outcome", "best_test_r2"]],
            left_on="outcome_column",
            right_on="outcome",
            how="inner",
        )
        .sort_values("r2_combined_observed", ascending=True)
        .reset_index(drop=True)
    )
    row_count = len(table)

    left_limits = (-0.35, 0.31)
    right_limits = (-0.35, 0.42)
    left_span = left_limits[1] - left_limits[0]
    right_span = right_limits[1] - right_limits[0]

    figure, (left_axes, right_axes) = plt.subplots(
        ncols=2,
        sharey=True,
        figsize=(7.16, 0.30 * row_count + 1.3),
        gridspec_kw={"width_ratios": [1.0, right_span / left_span]},
    )
    for axes in (left_axes, right_axes):
        axes.axvline(0.0, color=COLOUR_NEUTRAL, linewidth=0.8, zorder=2)

    for row_position, record in table.iterrows():
        left_axes.plot(
            [record["best_test_r2"], record["r2_combined_observed"]],
            [row_position, row_position],
            color=PALETTE_DARK,
            linewidth=1.5,
            solid_capstyle="round",
            zorder=4,
        )
        left_axes.plot(
            record["best_test_r2"],
            row_position,
            marker="o",
            markersize=5.4,
            markerfacecolor=PALETTE_DARK,
            markeredgecolor="white",
            markeredgewidth=0.7,
            zorder=5,
        )
        left_axes.plot(
            record["r2_combined_observed"],
            row_position,
            marker="D",
            markersize=4.8,
            markerfacecolor=COLOUR_INK,
            markeredgecolor="white",
            markeredgewidth=0.7,
            zorder=6,
        )

        right_axes.plot(
            [record["r2_covariate_only"], record["r2_combined_observed"]],
            [row_position, row_position],
            color=PALETTE_DARK,
            linewidth=1.5,
            solid_capstyle="round",
            zorder=4,
        )
        right_axes.plot(
            record["r2_covariate_only"],
            row_position,
            marker="o",
            markersize=4.8,
            markerfacecolor="white",
            markeredgecolor=COLOUR_NEUTRAL,
            markeredgewidth=1.0,
            zorder=5,
        )
        right_axes.plot(
            record["r2_combined_observed"],
            row_position,
            marker="o",
            markersize=5.4,
            markerfacecolor=PALETTE_DARKEST,
            markeredgecolor="white",
            markeredgewidth=0.7,
            zorder=6,
        )
        significance_mark = "*" if bool(record["reject_null"]) else ""
        right_axes.text(
            COMBINED_INCREMENT_COLUMN_RIGHT_EDGE,
            row_position,
            f"\u0394{record['observed_incremental_r2']:+.3f}{significance_mark}",
            fontsize=6.8,
            va="center",
            ha="right",
            color=COLOUR_INK,
        )

    left_axes.set_yticks(range(row_count))
    left_axes.set_yticklabels(
        [OUTCOME_DISPLAY_NAMES.get(name, name) for name in table["outcome_column"]],
        fontsize=7.2,
    )
    left_axes.set_ylim(-0.8, row_count + 0.3)
    left_axes.set_xlim(*left_limits)
    right_axes.set_xlim(*right_limits)
    for axes in (left_axes, right_axes):
        axes.set_xticks([-0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3])
        axes.set_xlabel("Held-out R²", fontsize=7.4)
        axes.tick_params(axis="x", labelsize=7.0)
        axes.spines["bottom"].set_bounds(-0.30, 0.30)
        axes.spines["bottom"].set_color(COLOUR_NEUTRAL_LIGHT)
        for spine_name in ("top", "right", "left"):
            axes.spines[spine_name].set_visible(False)
        axes.tick_params(axis="y", length=0)

    right_axes.text(
        COMBINED_INCREMENT_COLUMN_RIGHT_EDGE,
        row_count + 0.0,
        "Increment",
        fontsize=6.8,
        ha="right",
        color=COLOUR_NEUTRAL,
    )

    left_axes.legend(
        handles=[
            Line2D(
                [],
                [],
                marker="o",
                linestyle="none",
                markersize=5.4,
                markerfacecolor=PALETTE_DARK,
                markeredgecolor="white",
                label="48 months, best performing model",
            ),
            Line2D(
                [],
                [],
                marker="D",
                linestyle="none",
                markersize=4.8,
                markerfacecolor=COLOUR_INK,
                markeredgecolor="white",
                label="72 months, same model applied",
            ),
        ],
        loc="lower left",
        bbox_to_anchor=(0.0, 1.005),
        frameon=False,
        fontsize=7.0,
        handletextpad=0.4,
        borderpad=0.0,
    )
    right_axes.legend(
        handles=[
            Line2D(
                [],
                [],
                marker="o",
                linestyle="none",
                markersize=4.8,
                markerfacecolor="white",
                markeredgecolor=COLOUR_NEUTRAL,
                label="Covariates only",
            ),
            Line2D(
                [],
                [],
                marker="o",
                linestyle="none",
                markersize=5.4,
                markerfacecolor=PALETTE_DARKEST,
                markeredgecolor="white",
                label="Covariates and activity block",
            ),
        ],
        loc="lower left",
        bbox_to_anchor=(0.0, 1.005),
        frameon=False,
        fontsize=7.0,
        handletextpad=0.4,
        borderpad=0.0,
    )

    figure.tight_layout(w_pad=1.4)
    return _save_figure(
        figure=figure,
        output_directory=output_directory,
        file_stem=file_stem,
        margin_in_inches=0.16,
    )


def _save_figure(
    *,
    figure: plt.Figure,
    output_directory: Path,
    file_stem: str,
    margin_in_inches: float = 0.10,
) -> list[Path]:
    """Write one figure to two vector files and a raster file.

    :param figure: The figure to write.
    :param output_directory: Directory that will receive the rendered files.
    :param file_stem: Filename without extension.
    :param margin_in_inches: White or tinted margin kept around the content.
    :return: Paths of the written files, vector formats first (PDF,
        EPS, then PNG).
    """
    output_directory.mkdir(parents=True, exist_ok=True)
    vector_path = output_directory / f"{file_stem}.pdf"
    encapsulated_vector_path = output_directory / f"{file_stem}.eps"
    raster_path = output_directory / f"{file_stem}.png"
    figure.savefig(
        vector_path,
        bbox_inches="tight",
        pad_inches=margin_in_inches,
        facecolor="white",
    )
    figure.savefig(
        encapsulated_vector_path,
        format="eps",
        bbox_inches="tight",
        pad_inches=margin_in_inches,
        facecolor="white",
    )
    figure.savefig(
        raster_path,
        dpi=400,
        bbox_inches="tight",
        pad_inches=margin_in_inches,
        facecolor="white",
    )
    plt.close(figure)
    return [vector_path, encapsulated_vector_path, raster_path]

In [ ]:
apply_house_style()

figure_output = prediction_output / "figures"

written_figure_paths = draw_predictor_retention_figure(
    selection_frequency_path=prediction_output / "stage_1_feature_selection_frequency.csv",
    output_directory=figure_output,
    file_stem="figure_1_predictor_retention",
)
written_figure_paths += draw_transportability_figure(
    transportability_table=load_transportability_table(
        model_comparison_path=prediction_output / "stage_7_model_comparison.csv",
        longitudinal_validation_path=prediction_output / "stage_8_longitudinal_validation.csv",
    ),
    output_directory=figure_output,
    file_stem="figure_2_transportability_48_to_72_months",
)
written_figure_paths += draw_confirmatory_increment_figure(
    confirmatory_inference_path=prediction_output / "stage_9_confirmatory_inference.csv",
    output_directory=figure_output,
    file_stem="figure_3_covariates_versus_combined",
)
written_figure_paths += draw_combined_transport_and_increment_figure(
    model_comparison_path=prediction_output / "stage_7_model_comparison.csv",
    confirmatory_inference_path=prediction_output / "stage_9_confirmatory_inference.csv",
    output_directory=figure_output,
    file_stem="figure_2_and_3_combined",
)

for figure_path in written_figure_paths:
    print(figure_path)